In [ ]:
import h5py
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from plotly.colors import sample_colorscale
import re
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional, Union, List, Dict, Any
import plotly.express as px

In [ ]:
def load_simulation_h5(file_path: str) -> pd.DataFrame:
    """
    Custom loader for simulation HDF5 files with nested step groups.
    """
    data = []
    with h5py.File(file_path, "r") as f:
        sim_time = f.attrs.get("simulation_time", 0.0)
        for grp_name, grp in f.items():
            if not grp_name.startswith("step_"):
                continue

            attrs = dict(grp.attrs)
            for dset_name, dset in grp.items():
                attrs[dset_name] = dset[()]
            attrs["step"] = int(grp_name.split("_")[1])
            data.append(attrs)

    df = pd.DataFrame.from_records(data)
    df["simulation_time"] = sim_time
    return df.set_index("step").sort_index()


def get_simulation_data(
    study_path: Union[str, Path],
    order_by: Optional[str] = None,
    normalize_by: Optional[str] = None,
) -> pd.DataFrame:
    """
    Scans a study directory for data.h5 files and extracts parameters from folder names.
    Args:
        study_path: The root directory of the study to scrape.
        order_by: The parameter key (e.g., 'md') to sort the resulting DataFrame.
        normalize_by: The parameter key to use for a calculated 'norm_factor' column.
    Returns:
        A sorted Pandas DataFrame containing parameters, file paths, and metadata.
    """
    base_dir = Path(study_path)
    if not base_dir.exists():
        raise FileNotFoundError(f"Study path not found: {study_path}")

    file_paths: List[Path] = list(base_dir.rglob("data.h5"))
    data_list: List[Dict[str, Any]] = []

    param_regex = re.compile(r"([a-z]+)([-+]?[\d.]+(?:e[-+]?\d+)?)")

    for path in file_paths:
        folder_name: str = path.parent.name
        matches = param_regex.findall(folder_name)
        params: Dict[str, Any] = {key: float(val) for key, val in matches}

        params["path"] = str(path)
        params["folder_name"] = folder_name

        params["norm_factor"] = params.get(normalize_by, 1.0) if normalize_by else 1.0

        data_list.append(params)

    df = pd.DataFrame(data_list)

    if order_by and order_by in df.columns:
        df = df.sort_values(by=order_by).reset_index(drop=True)

    return df


def get_layout() -> go.Layout:
    """
    Returns a standardized Plotly layout for visualizations.
    """
    layout = go.Layout(
        xaxis=dict(
            showgrid=True,
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
            zeroline=False,
            ticks="inside",
            exponentformat="power",
            tickfont=dict(size=12),
        ),
        yaxis=dict(
            showgrid=True,
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
            zeroline=False,
            ticks="inside",
            exponentformat="power",
            tickfont=dict(size=12),
        ),
        font=dict(family="Latin-Modern", size=12, color="Black"),
        legend=dict(
            x=0.97,
            y=1.2,
            bgcolor="rgba(255, 255, 255, 0.8)",
            bordercolor="black",
            borderwidth=1,
            orientation="h",
            xanchor="right",
            yanchor="top",
        ),
        width=600,
        height=500,
        showlegend=True,
        template="plotly_white",
    )
    return layout

### General plots

In [3]:
study_name = "l_study_r1e-3_dd5e4_md5e5"
order_by = "l"
group_by = None
normalize_by = "l"

# study_name = "md_study_r1e0_dd5e4_l2e-2"
# order_by = "md"
# group_by = None
# normalize_by = None

# study_name = "dd_study_r1e0_l2e-2_md5e5"
# order_by = "dd"
# group_by = None
# normalize_by = None

# study_name = "dd_study_r1e-3_l2e-2_md5e5"
# order_by = "dd"
# group_by = None
# normalize_by = None

study_name = "r_study"
order_by = "r"
group_by = "seed"
normalize_by = "l"

# study_name = "r_study_penalty_1e-1"
# order_by = "r"
# group_by = "seed"
# normalize_by = "l"

# study_name = "r_study_penalty_1e0"
# order_by = "r"
# group_by = "seed"
# normalize_by = "l"

# study_name = "r_study_penalty_1e1"
# order_by = "r"
# group_by = "seed"
# normalize_by = "l"

study_name = "strain_rate_nsn_nobox"  # With 0.2*dtc
order_by = "r"
group_by = "seed"
normalize_by = "l"

# study_name = "strain_rate_pen_nobox" # With 0.2*dtc
# order_by = "r"
# group_by = "seed"
# normalize_by = "l"

study_name = "strain_rate_nsn_nobox_v2"  # With 0.99*dtc
order_by = "r"
group_by = "seed"
normalize_by = "l"

study_name = "time_study_no_box"
order_by = "s"
group_by = None
normalize_by = None

study_name = "simulation_time_box"
order_by = "s"
group_by = None
normalize_by = None

study_name = "mesh_convergence_box_r1e0_l2e-2"
order_by = "md"
group_by = None
normalize_by = None

# study_name = "timestep_convergence_box_r1e0_l2e-2"
# order_by = "s"
# group_by = None
# normalize_by = None

# study_name = "defect_convergence_box_r1e0_l2e-2"
# order_by = "dd"
# group_by = None
# normalize_by = None

study_name = "mesh_convergence_box_r1e0_l1e-2"
order_by = "md"
group_by = None
normalize_by = None

# study_name = "box_size_study_r1e0_l2e-2_md2e6"
# order_by = "bf"
# group_by = None
# normalize_by = None

# study_name = "restitution_study_r1e0_l2e-2_md2e6"
# order_by = "e"
# group_by = None
# normalize_by = None

# study_name = "impact_l1e-3_md2e6_cir5e-1"
# order_by = "k"
# group_by = None
# normalize_by = None

# study_name = "mesh_convergence_r1e1_box50"
# order_by = "md"
# group_by = None
# normalize_by = None

# study_name = "mesh_convergence_r1e1_box50_l2e-2"
# order_by = "md"
# group_by = None
# normalize_by = None

# study_name = "box_convergence_r1e1_md5e6"
# order_by = "bf"
# group_by = None
# normalize_by = "l"

# study_name = "test_contact_explicit"
# order_by = "md"
# group_by = None
# normalize_by = None

study_name = "length_study_r1e0_bf100_md2e6"
order_by = "l"
group_by = None
normalize_by = "l"

# study_name = "restitution_study_r1e0_bf100_md2e6"
# order_by = "e"
# group_by = None
# normalize_by = None

# study_name = "length_study_r1e0_bf100_md2e6_smallconvtol"
# order_by = "l"
# group_by = None
# normalize_by = "l"

# study_name = "restitution_study_r1e0_bf100_md2e6_smallconvtol"
# order_by = "e"
# group_by = None
# normalize_by = None

# study_name = "mesh_convergence_r1e0_l5e-3_bf100"
# order_by = "md"
# group_by = None
# normalize_by = None

# study_name = "length_convergence_r1e0_md1e6_bf100"
# order_by = "l"
# group_by = None
# normalize_by = "l"

# study_name = "restitution_r1e0_md1e6_l5e-3_bf100"
# order_by = "e"
# group_by = None
# normalize_by = None

In [ ]:
study_path = "../../output/cluster/" + study_name + "/"

df = get_simulation_data(study_path, order_by=order_by, normalize_by=normalize_by)

group_cols = [
    c for c in df.columns if c not in [group_by, "path", "folder_name", "norm_factor"]
]

print("Grouping by columns:", group_cols)
unique_configs = df.groupby(group_cols)
order_by_idx = group_cols.index(order_by)
sorted_groups = sorted(unique_configs.groups.items(), key=lambda x: x[0][order_by_idx])

colors = px.colors.sample_colorscale(
    "Viridis", [i / len(sorted_groups) for i in range(len(sorted_groups))]
)

Grouping by columns: ['nsnfrag', 'l', 'n', 'md', 'p', 't', 'r', 's', 'dd', 'e', 'sc', 'bf', 'seed']


In [11]:
fig_nbfrag = go.Figure()
fig_nbfrag_final = go.Figure()

# Lists to store data for the final plot
final_x_vals = []
final_y_vals = []
final_labels = []
final_colors = []

for (config_vals, group_indices), color in zip(sorted_groups, colors):
    subset = df.loc[group_indices]
    all_runs_data = []

    # Load all dataframes in this group
    for _, row in subset.iterrows():
        df_run = load_simulation_h5(row["path"])
        df_run["nb_fragments_norm"] = df_run["nb_fragments"] / row["norm_factor"]
        all_runs_data.append(df_run[["time", "nb_fragments_norm"]])

    combined = (
        pd.concat(all_runs_data)
        .groupby("time")["nb_fragments_norm"]
        .agg(["mean", "std"])
        .reset_index()
        .sort_values("time")
    )
    combined["std"] = combined["std"].fillna(0)

    # Determine label for this configuration
    current_val = subset.iloc[0][order_by]
    label = f"{order_by}={current_val:.2e}"

    # --- Plot 1: Time Series (fig_nbfrag) ---
    if group_by is not None:
        fig_nbfrag.add_trace(
            go.Scatter(
                x=pd.concat([combined["time"], combined["time"][::-1]]),
                y=pd.concat(
                    [
                        combined["mean"] + combined["std"],
                        (combined["mean"] - combined["std"])[::-1],
                    ]
                ),
                fill="toself",
                fillcolor=color.replace("rgb", "rgba").replace(")", ", 0.2)"),
                line=dict(color="rgba(255,255,255,0)"),
                hoverinfo="skip",
                showlegend=False,
            )
        )

    fig_nbfrag.add_trace(
        go.Scatter(
            x=combined["time"],
            y=combined["mean"],
            mode="lines",
            line=dict(color=color, width=2),
            name=label,
        )
    )

    # --- Collect data for Plot 2: Final Values (fig_nbfrag_final) ---
    final_row = combined.iloc[-1]
    final_x_vals.append(current_val)
    final_y_vals.append(final_row["mean"])
    final_labels.append(label)
    final_colors.append(color)

fig_nbfrag.layout = get_layout()
fig_nbfrag.update_layout(
    xaxis_title="Time (s)",
    yaxis_title="Fragments (Normalized)",
    legend=dict(x=1.4, orientation="h"),
    showlegend=True,
)

# --- Create the Final Value Plot ---
fig_nbfrag_final.add_trace(
    go.Scatter(
        x=final_x_vals,
        y=final_y_vals,
        mode="markers+lines",
        marker=dict(color=final_colors, size=10, symbol="circle"),
        line=dict(
            color="lightgray", dash="dash"
        ),  # Optional: line connecting the markers
        text=final_labels,
        hoverinfo="text+x+y",
    )
)

# Layout for Final Plot
fig_nbfrag_final.layout = get_layout()
fig_nbfrag_final.update_layout(
    title=f"Final Fragments Count vs {order_by}",
    xaxis_title=order_by,
    yaxis_title="Final Fragments (Normalized)",
    xaxis_type="linear",
    yaxis_type="log",
)

fig_nbfrag.show()
fig_nbfrag_final.show()

In [ ]:
fig_edis = go.Figure()
fig_edis_final = go.Figure()

# Lists to store data for the final plot
final_edis_x = []
final_edis_y = []
final_edis_std = []
final_labels = []
final_colors = []

for (config_vals, group_indices), color in zip(sorted_groups, colors):
    subset = df.loc[group_indices]
    all_runs_data = []

    # Load all dataframes in this group
    for _, row in subset.iterrows():
        df_run = load_simulation_h5(row["path"])
        df_run["dissipated_energy_norm"] = (
            df_run["dissipated_energy"] / row["norm_factor"]
        )
        # df_run["dissipated_energy_norm"] = (
        #    (df_run["dissipated_energy"]+df_run["contact_dissipation"]) / row["norm_factor"]
        # )
        all_runs_data.append(df_run[["time", "dissipated_energy_norm"]])

    combined = (
        pd.concat(all_runs_data)
        .groupby("time")["dissipated_energy_norm"]
        .agg(["mean", "std"])
        .reset_index()
        .sort_values("time")
    )
    combined["std"] = combined["std"].fillna(0)

    current_val = subset.iloc[0][order_by]
    label = f"{order_by}={current_val:.2e}"

    # --- Plot 1: Time Series (fig_edis) ---
    if group_by is not None:
        fig_edis.add_trace(
            go.Scatter(
                x=pd.concat([combined["time"], combined["time"][::-1]]),
                y=pd.concat(
                    [
                        combined["mean"] + combined["std"],
                        (combined["mean"] - combined["std"])[::-1],
                    ]
                ),
                fill="toself",
                fillcolor=color.replace("rgb", "rgba").replace(")", ", 0.2)"),
                line=dict(color="rgba(255,255,255,0)"),
                hoverinfo="skip",
                showlegend=False,
            )
        )

    fig_edis.add_trace(
        go.Scatter(
            x=combined["time"],
            y=combined["mean"],
            mode="lines",
            line=dict(color=color, width=2),
            name=label,
        )
    )

    # --- Collect data for Plot 2: Final Values ---
    final_row = combined.iloc[-1]
    final_edis_x.append(current_val)
    final_edis_y.append(final_row["mean"])
    final_edis_std.append(final_row["std"])
    final_labels.append(label)
    final_colors.append(color)

fig_edis.layout = get_layout()
fig_edis.update_layout(
    xaxis_title="Time (s)",
    yaxis_title="Dissipated Energy (Normalized)",
    legend=dict(x=1.4, orientation="h"),
    showlegend=True,
)

# --- Create the Final Value Plot ---
fig_edis_final.add_trace(
    go.Scatter(
        x=final_edis_x,
        y=final_edis_y,
        mode="markers+lines",
        marker=dict(color=final_colors, size=10),
        line=dict(color="rgba(150,150,150,0.5)", dash="dot"),
        error_y=dict(
            type="data", array=final_edis_std, visible=True
        ),  # Added error bars
        text=final_labels,
        hoverinfo="text+x+y",
    )
)

# Layout for Dissipated Energy Summary
fig_edis_final.layout = get_layout()
fig_edis_final.update_layout(
    title=f"Final Dissipated Energy vs {order_by}",
    xaxis_title=order_by,
    yaxis_title="Final Dissipated Energy (Normalized)",
    xaxis_type="linear",
    yaxis_type="log",
    template="plotly_white",
)

fig_edis.show()
fig_edis_final.show()

In [13]:
fig_etot = go.Figure()

# Lists to store data for the final plot
final_etot_x = []
final_etot_y = []
final_etot_std = []
final_labels = []
final_colors = []

for (config_vals, group_indices), color in zip(sorted_groups, colors):
    subset = df.loc[group_indices]
    all_runs_data = []

    # Load all dataframes in this group
    for _, row in subset.iterrows():
        df_run = load_simulation_h5(row["path"])
        df_run["algorithmic_energy_balance_norm"] = (
            df_run["algorithmic_energy_balance"]
            - df_run["algorithmic_energy_balance"].iloc[0]
        ) / row["norm_factor"]
        all_runs_data.append(df_run[["time", "algorithmic_energy_balance_norm"]])

    combined = (
        pd.concat(all_runs_data)
        .groupby("time")["algorithmic_energy_balance_norm"]
        .agg(["mean", "std"])
        .reset_index()
        .sort_values("time")
    )
    combined["std"] = combined["std"].fillna(0)

    current_val = subset.iloc[0][order_by]
    label = f"{order_by}={current_val:.2e}"

    # --- Plot 1: Time Series (fig_etot) ---
    if group_by is not None:
        fig_etot.add_trace(
            go.Scatter(
                x=pd.concat([combined["time"], combined["time"][::-1]]),
                y=pd.concat(
                    [
                        combined["mean"] + combined["std"],
                        (combined["mean"] - combined["std"])[::-1],
                    ]
                ),
                fill="toself",
                fillcolor=color.replace("rgb", "rgba").replace(")", ", 0.2)"),
                line=dict(color="rgba(255,255,255,0)"),
                hoverinfo="skip",
                showlegend=False,
            )
        )

    fig_etot.add_trace(
        go.Scatter(
            x=combined["time"],
            y=combined["mean"],
            mode="lines",
            line=dict(color=color, width=2),
            name=label,
        )
    )

    # --- Collect data for Plot 2: Final Values ---
    final_row = combined.iloc[-1]
    final_etot_x.append(current_val)
    final_etot_y.append(final_row["mean"])
    final_etot_std.append(final_row["std"])
    final_labels.append(label)
    final_colors.append(color)

fig_etot.layout = get_layout()
fig_etot.update_layout(
    xaxis_title="Time (s)",
    yaxis_title="Algorithmic Energy Balance (Normalized)",
    legend=dict(x=1.4, orientation="h"),
    showlegend=True,
)

fig_etot.show()

In [14]:
fig_sim_time = go.Figure()
for (config_vals, group_indices), color in zip(sorted_groups, colors):
    subset = df.loc[group_indices]
    simulation_times = []
    colors_new = []

    for _, row in subset.iterrows():
        df_run = load_simulation_h5(row["path"])
        simulation_time = df_run["simulation_time"].iloc[0]
        simulation_times.append(simulation_time / 60 / 60)  # Convert to hours

        if config_vals[-1] == -10.0:
            colors_new.append("red")
        else:
            colors_new.append("gray")

    current_val = subset.iloc[0][order_by]
    label = f"{order_by}={current_val:.2e}"
    fig_sim_time.add_trace(
        go.Scatter(
            x=[current_val] * len(simulation_times),
            y=simulation_times,
            mode="markers",
            marker=dict(color=colors_new, size=10),
            name=label,
        )
    )
fig_sim_time.layout = get_layout()
fig_sim_time.update_layout(
    title="Simulation Time vs " + order_by,
    xaxis_title=order_by,
    yaxis_title="Simulation Time (hours)",
    xaxis_type="log",
    yaxis_type="linear",
)
fig_sim_time.show()

In [9]:
import numpy as np
import plotly.graph_objects as go
import pandas as pd

final_data = []

for (config_vals, group_indices), color in zip(sorted_groups, colors):

    subset = df.loc[group_indices]
    final_nbfrags = []
    final_sizes = []
    final_edis = []

    # Load all dataframes in this group

    for _, row in subset.iterrows():

        df_run = load_simulation_h5(row["path"])

        # Normalize the target column
        df_run["nb_fragments_norm"] = df_run["nb_fragments"] / row["norm_factor"]

        final_nbfrags.append(df_run["nb_fragments_norm"].iloc[-1])
        final_sizes.append(row["norm_factor"] / df_run["nb_fragments"].iloc[-1])
        final_edis.append(df_run["dissipated_energy"].iloc[-1] / row["norm_factor"])

    mean_final = np.mean(final_nbfrags)
    std_final = np.std(final_nbfrags)

    mean_size = np.mean(final_sizes)
    std_size = np.std(final_sizes)

    mean_edis = np.mean(final_edis)
    std_edis = np.std(final_edis)

    final_data.append(
        {
            order_by: subset.iloc[0][order_by],
            "mean_nb_fragments": mean_final,
            "std_nb_fragments": std_final,
            "mean_fragment_size": mean_size,
            "std_fragment_size": std_size,
            "mean_dissipated_energy": mean_edis,
            "std_dissipated_energy": std_edis,
        }
    )

final_df = pd.DataFrame(final_data).sort_values(by=order_by)

# ==============================================================================
# 1. SETUP THEORETICAL MODELS
# ==============================================================================

# Material Properties
mat = dict(E=370e9, Gc=50, sigma_t=262e6, rho=3900)

# Derived scales
c = (mat["E"] / mat["rho"]) ** 0.5
s0 = mat["E"] * mat["Gc"] / mat["sigma_t"] ** 2
# Note: Ensure 'final_df[order_by]' is actually normalized strain rate.
# If it is raw strain rate (1/s), you must normalize it here:
# x_values = final_df[order_by] / (c * mat["sigma_t"]**3 / (mat["E"]**2 * mat["Gc"]))
# Otherwise, we assume it is already normalized:
x_grid = np.logspace(
    np.log10(final_df[order_by].min()), np.log10(final_df[order_by].max()), 300
)


# Theoretical Functions for Dimensionless Spacing s_bar
def sbar_grady(e):
    return (24.0 / (e**2)) ** (1.0 / 3.0)


def sbar_gc(e):
    return (4.0 / e) * np.sinh((1.0 / 3.0) * np.arcsinh(1.5 * e))


def sbar_zmr(e):
    return 4.5 / (1.0 + 4.5 * e ** (2.0 / 3.0))


# ---------------------------------------------------------
# A. Theoretical Fragment Size (s = s0 * s_bar)
# ---------------------------------------------------------
y_grady = s0 * sbar_grady(x_grid)
y_gc = s0 * sbar_gc(x_grid)
y_zmr = s0 * sbar_zmr(x_grid)

# ---------------------------------------------------------
# B. Theoretical Dissipated Energy
# ---------------------------------------------------------
# CAUTION: 'K' represents Total Energy in the formula below.
# If your plot 'mean_dissipated_energy' is normalized (e.g. Energy Density J/m^3),
# you must divide K by the Volume (A_eff * L_total).
A_eff = 1.0
L_total = 1.0  # Normalized Length
K_total = 2.0 * (mat["sigma_t"] ** 2) * A_eff * L_total / mat["E"]

# Assuming your plot is strictly Energy (J):
E_grady = K_total / sbar_grady(x_grid)
E_gc = K_total / sbar_gc(x_grid)
E_zmr = K_total / sbar_zmr(x_grid)

/tmp/ipykernel_7434/1029624925.py:65: RuntimeWarning:

divide by zero encountered in log10

/home/ghesquie/projects/lsms_codes/NS_Frag1D/.venv/lib/python3.11/site-packages/numpy/_core/function_base.py:162: RuntimeWarning:

invalid value encountered in multiply

/home/ghesquie/projects/lsms_codes/NS_Frag1D/.venv/lib/python3.11/site-packages/numpy/_core/function_base.py:172: RuntimeWarning:

invalid value encountered in add



In [10]:
# ==============================================================================
# 2. PLOT FIGURES
# ==============================================================================

# --- FIGURE 2: Mean Fragment Size (With Theory) ---
fig_size = go.Figure()

# Theory Curves
fig_size.add_trace(
    go.Scatter(
        x=x_grid,
        y=y_grady,
        mode="lines",
        name="Grady",
        line=dict(dash="dash", color="gray"),
    )
)
fig_size.add_trace(
    go.Scatter(
        x=x_grid,
        y=y_gc,
        mode="lines",
        name="Glenn-Chudnovsky",
        line=dict(dash="dot", color="black"),
    )
)
fig_size.add_trace(
    go.Scatter(
        x=x_grid,
        y=y_zmr,
        mode="lines",
        name="ZMR",
        line=dict(dash="dashdot", color="purple"),
    )
)

# Simulation Data
fig_size.add_trace(
    go.Scatter(
        x=final_df[order_by],
        y=final_df["mean_fragment_size"],
        error_y=dict(type="data", array=final_df["std_fragment_size"], visible=True),
        mode="markers",
        marker=dict(size=8, color="red"),
        line=dict(width=2),
        name="Simulation",
    )
)

fig_size.layout = get_layout()
fig_size.update_layout(
    title="Mean Fragment Size vs Strain Rate",
    xaxis_title=order_by,
    yaxis_title="Size (m)",
    xaxis_type="log",
    yaxis_type="log",
)
fig_size.show()


# --- FIGURE 3: Dissipated Energy (With Theory) ---
fig_edis_final = go.Figure()

# Theory Curves
fig_edis_final.add_trace(
    go.Scatter(
        x=x_grid,
        y=E_grady,
        mode="lines",
        name="Grady",
        line=dict(dash="dash", color="gray"),
    )
)
fig_edis_final.add_trace(
    go.Scatter(
        x=x_grid,
        y=E_gc,
        mode="lines",
        name="Glenn-Chudnovsky",
        line=dict(dash="dot", color="black"),
    )
)
fig_edis_final.add_trace(
    go.Scatter(
        x=x_grid,
        y=E_zmr,
        mode="lines",
        name="ZMR",
        line=dict(dash="dashdot", color="purple"),
    )
)

# Simulation Data
fig_edis_final.add_trace(
    go.Scatter(
        x=final_df[order_by],
        y=final_df["mean_dissipated_energy"],
        error_y=dict(
            type="data", array=final_df["std_dissipated_energy"], visible=True
        ),
        mode="markers",
        marker=dict(size=8, color="green"),
        line=dict(width=2),
        name="Simulation",
    )
)

fig_edis_final.layout = get_layout()
fig_edis_final.update_layout(
    title="Dissipated Energy vs Strain Rate",
    xaxis_title=order_by,
    yaxis_title="Energy (J)",
    xaxis_type="log",
    yaxis_type="log",
)
fig_edis_final.show()

### Specific plots

Strain rate study for both NSN and penalty-based contact.
- Contact penalty of $10\times E/\bar{h}_e$
- Cohesive stiffness cap at $10\times E/\bar{h}_e$

Both are set at $0.2\Delta t_c$, with $\Delta t_c$ taking into account the additional cohesive or contact stiffnesses.

Comparison with ZMR, Glenn & Chudnovsky and Grady.

In [11]:
write = False

In [12]:
study_names = ["strain_rate_pen_nobox", "strain_rate_nsn_nobox_v2"]  # With 0.2*dtc
marker_colors = ["rgba(0,0,0,0)", "red"]
marker_sizes = [8, 6]
outline_colors = ["rgba(40,99,119,1)", "rgba(0,0,0,0)"]
outline_sizes = [3, 0]
order_by = "r"
group_by = "seed"
normalize_by = "l"

dfs = {}
sorted_groups_all = {}

for study_name in study_names:
    # Load Data
    study_path = "../../output/cluster/" + study_name + "/"
    df = get_simulation_data(study_path, order_by=order_by, normalize_by=normalize_by)

    # Grouping Logic
    group_cols = [
        c
        for c in df.columns
        if c not in [group_by, "path", "folder_name", "norm_factor"]
    ]
    unique_configs = df.groupby(group_cols)
    order_by_idx = group_cols.index(order_by)
    sorted_groups = sorted(
        unique_configs.groups.items(), key=lambda x: x[0][order_by_idx]
    )

    # Store results
    dfs[study_name] = df
    sorted_groups_all[study_name] = sorted_groups

# Compute ZMR, Grady, and Glenn-Chudnovsky theoretical predictions
# Material Properties
mat = dict(E=370e9, Gc=50, sigma_c=262e6, rho=3900)

# Derived scales
c = (mat["E"] / mat["rho"]) ** 0.5
s0 = mat["E"] * mat["Gc"] / mat["sigma_c"] ** 2
U0 = mat["sigma_c"] ** 2 / mat["E"]


# Theoretical Functions for Dimensionless Fragment Size s_bar
def sbar_grady(e: float) -> float:
    return (24.0 / (e**2)) ** (1.0 / 3.0)


def sbar_gc(e: float) -> float:
    return (4.0 / e) * np.sinh((1.0 / 3.0) * np.arcsinh(1.5 * e))


def sbar_zmr(e: float) -> float:
    return 4.5 / (1.0 + 4.5 * e ** (2.0 / 3.0))


# Normalized strain rate grid
x_grid = np.logspace(-3.5, 3.5, 400)

# Normalized Fragment Size Predictions
y_grady = sbar_grady(x_grid)
y_gc = sbar_gc(x_grid)
y_zmr = sbar_zmr(x_grid)

# Dissipated Energy Predictions
E_grady = 1 / sbar_grady(x_grid)  # Crack density (number of fragments per unit length)
E_gc = 1 / sbar_gc(x_grid)
E_zmr = 1 / sbar_zmr(x_grid)

In [ ]:
write = True

fig_combined = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("", ""),
    horizontal_spacing=0.02,
    shared_xaxes=True,
)


# Size Theory (Column 1)
fig_combined.add_trace(
    go.Scatter(
        x=x_grid,
        y=y_zmr,
        name="ZMR Theory",
        line=dict(dash="dash", color="black"),
        legendgroup="theory",
        showlegend=True,
    ),
    row=1,
    col=1,
)
fig_combined.add_trace(
    go.Scatter(
        x=x_grid,
        y=y_gc,
        name="Glenn-Chudnovsky Theory",
        line=dict(dash="dot", color="gray"),
        legendgroup="theory",
        showlegend=True,
    ),
    row=1,
    col=1,
)
fig_combined.add_trace(
    go.Scatter(
        x=x_grid,
        y=y_grady,
        name="Grady Theory",
        line=dict(dash="dashdot", color="lightgray"),
        legendgroup="theory",
        showlegend=True,
    ),
    row=1,
    col=1,
)

# Energy Theory (Column 2)
fig_combined.add_trace(
    go.Scatter(
        x=x_grid,
        y=E_zmr,
        name="ZMR Theory",
        line=dict(dash="dash", color="black"),
        legendgroup="theory",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig_combined.add_trace(
    go.Scatter(
        x=x_grid,
        y=E_gc,
        name="Glenn-Chudnovsky Theory",
        line=dict(dash="dot", color="gray"),
        legendgroup="theory",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig_combined.add_trace(
    go.Scatter(
        x=x_grid,
        y=E_grady,
        name="Grady Theory",
        line=dict(dash="dashdot", color="lightgray"),
        legendgroup="theory",
        showlegend=False,
    ),
    row=1,
    col=2,
)

for study_name, marker_color, marker_size, outline_size, outline_color in zip(
    study_names, marker_colors, marker_sizes, outline_sizes, outline_colors
):
    df = dfs[study_name]
    sorted_groups = sorted_groups_all[study_name]
    final_data = []

    for config_vals, group_indices in sorted_groups:
        subset = df.loc[group_indices]
        final_nbfrags, final_sizes, final_edis = [], [], []

        for _, row in subset.iterrows():
            df_run = load_simulation_h5(row["path"])
            final_nbfrags.append(df_run["nb_fragments"].iloc[-1] / row["norm_factor"])
            final_sizes.append(row["norm_factor"] / df_run["nb_fragments"].iloc[-1])
            final_edis.append(df_run["dissipated_energy"].iloc[-1] / row["norm_factor"])

        final_data.append(
            {
                order_by: subset.iloc[0][order_by],
                "mean_size": np.mean(final_sizes),
                "mean_edis": np.mean(final_edis),
            }
        )

    final_df = pd.DataFrame(final_data).sort_values(by=order_by)

    # Plot Size Data (Col 1)
    fig_combined.add_trace(
        go.Scatter(
            x=final_df[order_by],
            y=final_df["mean_size"] / s0,
            mode="markers",
            name=study_name,
            legendgroup=study_name,
            marker=dict(
                size=marker_size,
                color=marker_color,
                line=dict(width=outline_size, color=outline_color),
            ),
        ),
        row=1,
        col=1,
    )

    # Plot Energy Data (Col 2)
    fig_combined.add_trace(
        go.Scatter(
            x=final_df[order_by],
            y=final_df["mean_edis"] / U0,
            mode="markers",
            name=study_name,
            legendgroup=study_name,
            showlegend=False,
            marker=dict(
                size=marker_size,
                color=marker_color,
                line=dict(width=outline_size, color=outline_color),
            ),
        ),
        row=1,
        col=2,
    )

fig_combined.update_layout(get_layout())

# Define common axis styling from your original layout
axis_style = dict(
    showgrid=True,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    exponentformat="power",
    type="log",
    tickmode="linear",
    dtick=1,
    tickfont=dict(size=10),
)

# Apply to all subplots
fig_combined.update_xaxes(axis_style)
fig_combined.update_yaxes(axis_style)

# Set specific plots attributes
fig_combined.update_xaxes(title_text=r"$\hat{\dot\varepsilon}$")  # Sets for both
fig_combined.update_yaxes(title_text=r"$\hat{s}$", row=1, col=1)
fig_combined.update_yaxes(title_text=r"$\hat{\mathcal{G}}$", row=1, col=2, side="right")

fig_combined.update_layout(
    width=700,
    height=350,
    showlegend=False,
)

fig_combined.show()

if write:
    fig_combined.write_image(
        "../../output/figures/strain_rate_nsn_pen_combined_plot_differentdt_new.pdf",
        scale=1,
    )

Comparison of the two methods (nsn vs. penalty) at their respective stability limits (respectively $0.99$ and $0.2\Delta t_c$) in terms of:
- Evolution of the number of fragments
- Evolution of the algorithmic energy.

In [14]:
write = False

In [15]:
run_names = [
    "output/cluster/time_study_no_box/BD-ns_frag1d-runs/run-221/nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_k1.00e+01_bc_seed1",
    "output/cluster/time_study_no_box/BD-ns_frag1d-runs/run-219/nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s5.00e-01_dd1.00e+05_k1.00e+01_bc_seed1",
    "output/cluster/time_study_no_box/BD-ns_frag1d-runs/run-391/nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s4.00e-01_dd1.00e+05_k1.00e+01_bc_seed1",
    "output/cluster/time_study_no_box/BD-ns_frag1d-runs/run-390/nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s3.00e-01_dd1.00e+05_k1.00e+01_bc_seed1",
    "output/cluster/time_study_no_box/BD-ns_frag1d-runs/run-217/nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s2.00e-01_dd1.00e+05_k1.00e+01_bc_seed1",
    "output/cluster/time_study_no_box/BD-ns_frag1d-runs/run-222/nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_seed1",
    # "output/cluster/time_study_no_box/BD-ns_frag1d-runs/run-214/nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s1.00e-01_dd1.00e+05_k1.00e+01_bc_seed1"
]

line_colors = [
    "rgba(40,99,119,0.4)",
    "rgba(40,99,119,0.4)",
    "rgba(40,99,119,0.6)",
    "rgba(40,99,119,0.8)",
    "rgba(40,99,119,1)",
    "red",
]

for run_name in run_names:
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    sim_time = df_run["simulation_time"].iloc[0]
    print(
        f"Simulation Time (hours): {sim_time/3600:.2f} for run: {run_name.split('/')[-1]}"
    )

Simulation Time (hours): 0.28 for run: nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_k1.00e+01_bc_seed1
Simulation Time (hours): 0.00 for run: nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s5.00e-01_dd1.00e+05_k1.00e+01_bc_seed1
Simulation Time (hours): 0.00 for run: nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s4.00e-01_dd1.00e+05_k1.00e+01_bc_seed1
Simulation Time (hours): 10.42 for run: nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s3.00e-01_dd1.00e+05_k1.00e+01_bc_seed1
Simulation Time (hours): 9.40 for run: nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s2.00e-01_dd1.00e+05_k1.00e+01_bc_seed1
Simulation Time (hours): 13.19 for run: nsnfrag1d_l1.00e-01_n5.00e+04_md5.00e+05_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_seed1


In [16]:
fig_combined = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=("", ""),
    vertical_spacing=0.04,
    shared_xaxes=True,
)

for run_name, line_color in zip(run_names, line_colors):
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    fig_combined.add_trace(
        go.Scatter(
            x=df_run["time"],
            y=df_run["nb_fragments"],
            mode="lines",
            line=dict(color=line_color, width=2),
            name=run_name.split("/")[-1],
        ),
        row=1,
        col=1,
    )
    fig_combined.add_trace(
        go.Scatter(
            x=df_run["time"],
            y=(
                df_run["algorithmic_energy_balance"]
                - df_run["algorithmic_energy_balance"].iloc[0]
            )
            / df_run["algorithmic_energy_balance"].iloc[0],
            mode="lines",
            line=dict(color=line_color, width=2),
            name=run_name.split("/")[-1],
            showlegend=False,
        ),
        row=2,
        col=1,
    )

axis_style = dict(
    showgrid=True,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    exponentformat="power",
    showexponent="last",
)

fig_combined.update_layout(get_layout())

# Apply to all subplots
fig_combined.update_xaxes(axis_style, range=[0, 4e-6])
fig_combined.update_yaxes(axis_style, row=2, col=1)

# Set specific plots attributes
fig_combined.update_xaxes(title_text="", row=1, col=1)
fig_combined.update_yaxes(title_text=r"$N_f$", row=1, col=1, range=[390, 460])
fig_combined.update_xaxes(title_text="Time (s)", row=2, col=1)
fig_combined.update_yaxes(
    title_text=r"$\Delta \mathcal{H}/\mathcal{H}_0$", row=2, col=1, range=[-1e-6, 7e-6]
)
fig_combined.update_layout(
    width=400,
    height=400,
    showlegend=False,
)
fig_combined.show()

if write:
    fig_combined.write_image(
        "../../output/figures/time_study_no_box_nbfrag_etot_plot.pdf", scale=1
    )

Internally damaged impacting bar.

1. Position, velocity and contact force for internally damaged impacting bar, comparing:
    - Penalty-based for $\hat{k}^- \in[10^{-3}, 10^{3}]$ with $\Delta t = 0.1\Delta t_c$
    - Nonsmooth with $\tilde{k}=10E/h$ with $\Delta = 0.99 \Delta t_c$ and $e\in[0,1]$

2. Error plot $\eta_u$ for penalties vs. nonsmooth

2. Simulation time 

Note: $L=10^{-3}$ m, $\rho_\text{el}=2 \times 10^6$, $v_\text{impact}=5$ m/s, $\rho_\text{coh}=0.5$, $T=10^{-6}$ s.

In [17]:
run_names = [
    # PENALTIES
    "output/cluster/impact_full_study/BD-nsn_impact_study-runs/run-47/nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s1.00e-01_dd1.00e+05_k1.00e-02_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact",
    "output/cluster/impact_full_study/BD-nsn_impact_study-runs/run-57/nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s1.00e-01_dd1.00e+05_k1.00e-01_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact",
    "output/cluster/impact_full_study/BD-nsn_impact_study-runs/run-67/nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s1.00e-01_dd1.00e+05_k1.00e+00_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact",
    "output/cluster/impact_full_study/BD-nsn_impact_study-runs/run-77/nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s1.00e-01_dd1.00e+05_k1.00e+01_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact",
    "output/cluster/impact_full_study/BD-nsn_impact_study-runs/run-87/nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s1.00e-01_dd1.00e+05_k1.00e+02_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact",
    # NSN
    "output/cluster/impact_full_study/BD-nsn_impact_study-runs/run-32/nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s5.00e-01_dd1.00e+05_e0.000e+00_sc-1.00e+00_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact",
]


colorscale = "Darkmint_r"
line_colors = sample_colorscale(colorscale, len(run_names))  # +1 to adjust last color
line_colors = line_colors[:-1][
    ::-1
]  # Remove last two colors to avoid very light colors
line_colors.append("red")

print(line_colors)

for run_name in run_names:
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    sim_time = df_run["simulation_time"].iloc[0]
    print(
        f"Simulation Time (hours): {sim_time/3600:.2f} for run: {run_name.split('/')[-1]}"
    )

['rgb(157, 213, 190)', 'rgb(108, 175, 169)', 'rgb(69, 137, 145)', 'rgb(40, 99, 119)', 'rgb(18, 63, 90)', 'red']
Simulation Time (hours): 0.31 for run: nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s1.00e-01_dd1.00e+05_k1.00e-02_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact
Simulation Time (hours): 0.31 for run: nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s1.00e-01_dd1.00e+05_k1.00e-01_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact
Simulation Time (hours): 0.39 for run: nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s1.00e-01_dd1.00e+05_k1.00e+00_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact
Simulation Time (hours): 0.44 for run: nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s1.00e-01_dd1.00e+05_k1.00e+01_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact
Simulation Time (hours): 1.04 for run: nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s1.00e-01_dd1.00e+05_k1.00e+02_iv5.00e+00_bc_box_bf1.

In [18]:
df_run = load_simulation_h5("../../" + run_names[-1] + "/data.h5")

# Material and Geometric Properties
L = 1e-3
E, rho = 370e9, 3900
c = (E / rho) ** 0.5

# Analytical Impact and Rebound Times
t_analytic = df_run["time"]
u_ini = df_run["impact_position"].iloc[0]
v_ini = df_run["impact_velocity"].iloc[0]
t_impact = -u_ini / v_ini
t_rebound = t_impact + 2 * L / c


t_values = t_analytic.values
u_analytic = np.piecewise(
    t_values,
    [
        t_values < t_impact,
        (t_values >= t_impact) & (t_values < t_rebound),
        t_values >= t_rebound,
    ],
    [
        lambda t: (u_ini + v_ini * t) * (-1),
        lambda t: 0.0,
        lambda t: v_ini * (t - t_rebound),
    ],
)
v_analytic = np.piecewise(
    t_values,
    [
        t_values < t_impact,
        (t_values >= t_impact) & (t_values < t_rebound),
        t_values >= t_rebound,
    ],
    [
        lambda t: v_ini * (-1),
        lambda t: 0.0,
        lambda t: v_ini,
    ],
)
f_analytic = np.piecewise(
    t_values,
    [
        t_values < t_impact,
        (t_values >= t_impact) & (t_values < t_rebound),
        t_values >= t_rebound,
    ],
    [
        lambda t: 0.0,
        lambda t: rho * c * 1.0 * (v_ini),
        lambda t: 0.0,
    ],
)

In [19]:
write = False

fig_pos = go.Figure()

fig_pos.add_trace(
    go.Scatter(
        x=(t_analytic - t_impact) / (t_rebound - t_impact),
        y=u_analytic / L,
        mode="lines",
        line=dict(dash="solid", color="black", width=2),
        name="Analytical",
    ),
)

for run_name, line_color in zip(run_names, line_colors):
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    fig_pos.add_trace(
        go.Scatter(
            x=(df_run["time"] - t_impact) / (t_rebound - t_impact),
            y=df_run["impact_position"] * (-1) / L,
            mode="lines",
            line=dict(color=line_color, width=2),
            name=run_name.split("/")[-1],
        ),
    )

axis_style = dict(
    showgrid=True,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    exponentformat="power",
    showexponent="last",
)

fig_pos.update_layout(get_layout())
fig_pos.update_xaxes(axis_style)
fig_pos.update_yaxes(axis_style)
fig_pos.update_layout(
    xaxis_title=r"$t/t_b$ (s)",
    yaxis_title=r"$u_c/L$",
    width=600,
    height=400,
    showlegend=False,
)
fig_pos.show()

if write:
    fig_pos.write_image(
        "../../output/figures/impact_position_vs_time_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

fig_pos.update_xaxes(range=[-0.1, 1.1])
fig_pos.update_yaxes(range=[-7e-6, 2e-6], side="right")
fig_pos.show()

if write:
    fig_pos.write_image(
        "../../output/figures/impact_position_vs_time_plot_zoomed.pdf",
        scale=1,
        width=280,
        height=250,
    )

fig_vel = go.Figure()

fig_vel.add_trace(
    go.Scatter(
        x=(t_analytic - t_impact) / (t_rebound - t_impact),
        y=v_analytic / v_ini,
        mode="lines",
        line=dict(dash="solid", color="black", width=2),
        name="Analytical",
    ),
)

for run_name, line_color in zip(run_names, line_colors):
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    fig_vel.add_trace(
        go.Scatter(
            x=(df_run["time"] - t_impact) / (t_rebound - t_impact),
            y=df_run["impact_velocity"] * (-1) / v_ini,
            mode="lines",
            line=dict(color=line_color, width=2),
            name=run_name.split("/")[-1],
        ),
    )

fig_vel.update_layout(get_layout())
fig_vel.update_xaxes(axis_style)
fig_vel.update_yaxes(axis_style)
fig_vel.update_layout(
    xaxis_title=r"$t/t_b$ (s)",
    yaxis_title=r"$v_c/v_0$ (m/s)",
    width=600,
    height=400,
    showlegend=False,
)
fig_vel.show()

if write:
    fig_vel.write_image(
        "../../output/figures/impact_velocity_vs_time_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

fig_force = go.Figure()

fig_force.add_trace(
    go.Scatter(
        x=(t_analytic - t_impact) / (t_rebound - t_impact),
        y=f_analytic,
        mode="lines",
        line=dict(dash="solid", color="black", width=2),
        name="Analytical",
    ),
)

for run_name, line_color in zip(run_names, line_colors):
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    fig_force.add_trace(
        go.Scatter(
            x=(df_run["time"] - t_impact) / (t_rebound - t_impact),
            y=df_run["impact_force"],
            mode="lines",
            line=dict(color=line_color, width=2),
            name=run_name.split("/")[-1],
        ),
    )
fig_force.update_layout(get_layout())
fig_force.update_xaxes(axis_style)
fig_force.update_yaxes(axis_style)
fig_force.update_layout(
    xaxis_title=r"$t/t_b$ (s)",
    yaxis_title=r"$F_c$ (N)",
    width=600,
    height=400,
    showlegend=False,
)
fig_force.show()

if write:
    fig_force.write_image(
        "../../output/figures/impact_force_vs_time_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

In [20]:
fig_ealgo = go.Figure()
for run_name, line_color in zip(run_names, line_colors):
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    fig_ealgo.add_trace(
        go.Scatter(
            x=df_run["time"],
            y=(
                df_run["algorithmic_energy_balance"]
                - df_run["algorithmic_energy_balance"].iloc[0]
            )
            / df_run["algorithmic_energy_balance"].iloc[0],
            mode="lines",
            line=dict(color=line_color, width=2),
            name=run_name.split("/")[-1],
        ),
    )
fig_ealgo.update_layout(get_layout())
fig_ealgo.update_xaxes(axis_style)
fig_ealgo.update_yaxes(axis_style)
fig_ealgo.update_layout(
    xaxis_title=r"$t$ (s)",
    yaxis_title=r"$\Delta \mathcal{H}/\mathcal{H}_0$",
    width=600,
    height=400,
    showlegend=False,
)
fig_ealgo.show()

if write:
    fig_ealgo.write_image(
        "../../output/figures/impact_energy_balance_vs_time_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

Internally damaged impacting bar.

1. Error plot $\eta_u$ and $\eta_v$ for penalties vs. nonsmooth

2. Simulation time 

Note: $L=10^{-3}$ m, $\rho_\text{el}=2 \times 10^6$, $v_\text{impact}=5$ m/s, $\rho_\text{coh}=0.5$, $T=10^{-6}$ s.

In [ ]:
study_name = "impact_full_study"
study_path = "../../output/cluster/" + study_name + "/"

data = get_simulation_data(study_path=study_path)
data["k"] = data["k"].to_numpy(dtype=float)

df_run = load_simulation_h5(data["path"].iloc[0])

# Material and Geometric Properties
L = 1e-3
E, rho = 370e9, 3900
c = (E / rho) ** 0.5

# Analytical Impact and Rebound Times
t_analytic = df_run["time"]
u_ini = df_run["impact_position"].iloc[0]
v_ini = df_run["impact_velocity"].iloc[0]
t_impact = -u_ini / v_ini
t_rebound = t_impact + 2 * L / c


dt_crit_bulk = 1.13e-11
dt_crits = [1.13e-11, 1.13e-11, 1.07e-11, 7.81e-12, 3.27e-12, 1.07e-11]

colorscale = "Darkmint_r"
line_colors = sample_colorscale(colorscale, len(dt_crits) + 2)
line_colors = line_colors[:-1][::-1]
line_colors = line_colors[2:]
line_colors += ["red"]
line_colors = [
    "rgb(157, 213, 190)",
    "rgb(108, 175, 169)",
    "rgb(69, 137, 145)",
    "rgb(40, 99, 119)",
    "rgb(18, 63, 90)",
    "red",
]


# Exclude certain (k, s) pairs if the simulation was cut because unstable.
exclusions = {
    0.01: [0.99, 0.01],
    0.1: [0.99, 0.01],
    1.0: [0.01],
    10.0: [0.99, 0.9, 0.8, 0.7, 0.01],
    100.0: [0.99, 0.9, 0.8, 0.7, 0.01],
    np.nan: [],
}


def relative_error_l1(sim: np.ndarray, ana: np.ndarray) -> float:
    numerator = np.abs(np.sum((sim - ana)))
    denominator = np.abs(np.sum(ana))
    if denominator == 0:
        return 0.0 if numerator == 0 else np.inf
    return numerator / denominator


def relative_error_l2(sim: np.ndarray, ana: np.ndarray) -> float:
    numerator = np.sqrt(np.sum((sim - ana) ** 2))
    denominator = np.sqrt(np.sum(ana**2))
    if denominator == 0:
        return 0.0 if numerator == 0 else np.inf
    return numerator / denominator


k_unique = [0.01, 0.1, 1.0, 10.0, 100.0, np.nan]
k_to_dt_crit = dict(zip(k_unique, dt_crits))

In [22]:
axis_style = dict(
    showgrid=True,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    exponentformat="power",
    showexponent="last",
)

In [ ]:
calculated_data = {}

print("Starting batch processing...")

for i, (k_val, group) in enumerate(data.groupby("k", dropna=False)):
    print(f"Processing group k: {k_val:.2e}")

    # --- Filtering & Sorting ---
    current_exclusions = []
    for key, values in exclusions.items():
        if (pd.isna(key) and pd.isna(k_val)) or (key == k_val):
            current_exclusions = values
            break

    if current_exclusions:
        group = group[~group["s"].isin(current_exclusions)]

    group_sorted = group.sort_values("s")

    # Calculate x-axis values for this group
    current_dt_crit = (
        k_to_dt_crit.get(k_val) if pd.notna(k_val) else k_to_dt_crit[np.nan]
    )
    dt_values = group_sorted["s"].values * current_dt_crit

    # Initialize lists for this group
    pos_errors = []
    vel_errors = []
    instability_metrics = []
    cpu_times = []

    # --- Inner Loop: Load File ONCE per simulation ---
    for _, row in group_sorted.iterrows():
        df_run = load_simulation_h5(row["path"])
        t_values = df_run["time"].values

        # 1. Prepare Analytical Masks (Used for Pos & Vel)
        post_impact_mask = t_values > t_rebound
        t_post = t_values[post_impact_mask]

        # A. CPU Time (Hours)
        sim_time_seconds = df_run["simulation_time"].iloc[0]
        cpu_times.append(sim_time_seconds / 3600.0)

        # --- METRIC A: Position Error ---
        sim_pos_post = df_run["impact_position"].values[post_impact_mask] * (-1)
        u_analytic = np.piecewise(
            t_post, [t_post >= t_rebound], [lambda t: v_ini * (t - t_rebound)]
        )
        pos_errors.append(relative_error_l2(sim_pos_post, u_analytic))

        # --- METRIC B: Velocity Error ---
        sim_vel_post = df_run["impact_velocity"].values[post_impact_mask] * (-1)
        v_analytic = np.piecewise(t_post, [t_post >= t_rebound], [lambda t: v_ini])
        vel_errors.append(relative_error_l2(sim_vel_post, v_analytic))

        # --- METRIC C: Instability Ratio ---
        h_balance = df_run["algorithmic_energy_balance"].values
        h0 = h_balance[0]
        max_delta_h = np.max(np.abs(h_balance - h0))
        rel_energy_balance = max_delta_h / np.abs(h0)
        print(
            f"Max relative energy change for run {row['path']}: {rel_energy_balance:.2e}"
        )
        instability_metrics.append(rel_energy_balance)

        # max_e_int = np.max(
        #    df_run["potential_energy"]
        #    + df_run["reversible_energy"]
        #    + df_run["contact_energy"]
        # )
        # if max_e_int == 0:
        #    max_e_int = 1e-9  # Prevent div/0
    #
    # error_e = (
    #    df_run["algorithmic_energy_balance"]
    #    - df_run["algorithmic_energy_balance"].iloc[0]
    # )
    # max_instability = np.max(np.abs(error_e / max_e_int))
    # instability_metrics.append(max_instability)

    # Store everything for this k_val
    calculated_data[k_val] = {
        "dt_norm": dt_values / dt_crit_bulk,
        "pos_err": pos_errors,
        "vel_err": vel_errors,
        "instability": instability_metrics,
        "cpu_time": cpu_times,
        "color_idx": i,
    }

print("Computation complete. Generating plots...")

k_vals_to_process = [0.01, 0.1, 1.0, 10.0, 100.0, np.nan]


def add_traces_to_fig(fig, data_dict, y_key, x_key="dt_norm", show_dt_lines=False):
    """
    Generic helper to add traces for any stored metric.
    x_key defaults to "dt_norm", but can be set to "cpu_time".
    """
    for k_val in k_vals_to_process:
        if k_val not in data_dict:
            if pd.isna(k_val):
                nan_keys = [k for k in data_dict.keys() if pd.isna(k)]
                if not nan_keys:
                    continue
                k_val = nan_keys[0]
            else:
                continue

        res = data_dict[k_val]

        idx = res.get(
            "color_idx", k_vals_to_process.index(k_val) if not pd.isna(k_val) else -1
        )
        current_color = line_colors[idx % len(line_colors)]

        label = f"k={k_val:.2e}" if pd.notna(k_val) else "nonsmooth"

        fig.add_trace(
            go.Scatter(
                x=res[x_key],
                y=res[y_key],
                mode="lines+markers",
                line=dict(color=current_color, width=2),
                marker=dict(size=5, color=current_color),
                name=label,
            )
        )

    if show_dt_lines:
        for dt_c in dt_crits:
            fig.add_vline(
                x=dt_c / dt_crit_bulk,
                line_dash="solid",
                line_color="gray",
                line_width=1,
            )

Starting batch processing...
Processing group k: 1.00e-02
Max relative energy change for run ../../output/cluster/impact_full_study/BD-nsn_impact_study-runs/run-110/nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s2.00e-02_dd1.00e+05_k1.00e-02_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact/data.h5: 2.67e-07
Max relative energy change for run ../../output/cluster/impact_full_study/BD-nsn_impact_study-runs/run-111/nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s5.00e-02_dd1.00e+05_k1.00e-02_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact/data.h5: 6.66e-07
Max relative energy change for run ../../output/cluster/impact_full_study/BD-nsn_impact_study-runs/run-47/nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t1.00e-06_r1.00e+00_s1.00e-01_dd1.00e+05_k1.00e-02_iv5.00e+00_bc_box_bf1.00e+01_seed1_cir0.5_impact/data.h5: 1.58e-06
Max relative energy change for run ../../output/cluster/impact_full_study/BD-nsn_impact_study-runs/run-48/nsnfrag1d_l1.00e-03_n2.00e+03_

In [ ]:
# --- Plot 1: Position Error ---
write = False

fig_error_pos = go.Figure()
add_traces_to_fig(
    fig_error_pos, calculated_data, y_key="pos_err", x_key="dt_norm", show_dt_lines=True
)
fig_error_pos.update_layout(get_layout())
fig_error_pos.update_xaxes(axis_style, type="log")
fig_error_pos.update_yaxes(axis_style, type="log", side="right")
fig_error_pos.update_layout(
    title="Relative Error in Impact Position",
    xaxis_title=r"$\Delta t / \Delta t_{c, \text{bulk}}$",
    yaxis_title=r"$\eta_{u,2}$",
    width=600,
    height=400,
    showlegend=False,
)
fig_error_pos.show()
if write:
    fig_error_pos.write_image(
        "../../output/figures/impact_position_relative_l2_error_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

In [ ]:
# --- Plot 2: Velocity Error ---
write = False

fig_error_vel = go.Figure()
add_traces_to_fig(
    fig_error_vel, calculated_data, y_key="vel_err", x_key="dt_norm", show_dt_lines=True
)
fig_error_vel.update_layout(get_layout())
fig_error_vel.update_xaxes(axis_style, type="log")
fig_error_vel.update_yaxes(axis_style, type="log", side="right")
fig_error_vel.update_layout(
    title="Relative Error in Impact Velocity",
    xaxis_title=r"$\Delta t / \Delta t_{c, \text{bulk}}$",
    yaxis_title=r"$\eta_{v,2}$",
    width=600,
    height=400,
    showlegend=False,
)
fig_error_vel.show()
if write:
    fig_error_vel.write_image(
        "../../output/figures/impact_velocity_relative_l2_error_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

In [ ]:
# --- Plot 3: Instability ---
write = False

fig_instability = go.Figure()
add_traces_to_fig(
    fig_instability,
    calculated_data,
    y_key="instability",
    x_key="dt_norm",
    show_dt_lines=True,
)
fig_instability.update_layout(get_layout())
fig_instability.update_xaxes(axis_style, type="log")
fig_instability.update_yaxes(axis_style, type="log", side="left")
fig_instability.update_layout(
    title="Maximum Energy Instability Ratio",
    xaxis_title=r"$\Delta t / \Delta t_{c, \text{bulk}}$",
    yaxis_title=r"$\max(|\Delta E / E_{\text{int,max}}|)$",
    width=600,
    height=400,
    showlegend=False,
)
fig_instability.show()
if write:
    fig_instability.write_image(
        "../../output/figures/impact_energy_instability_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

In [ ]:
# --- Plot 4: Position Error vs CPU Time ---
write = False

fig_cpu_pos = go.Figure()
add_traces_to_fig(
    fig_cpu_pos, calculated_data, y_key="pos_err", x_key="cpu_time", show_dt_lines=False
)
fig_cpu_pos.update_layout(get_layout())
fig_cpu_pos.update_xaxes(
    axis_style, type="log"
)  # Usually linear for time, but user can change to "log" if desired
fig_cpu_pos.update_yaxes(axis_style, type="log", side="right")
fig_cpu_pos.update_layout(
    title="CPU Time vs Position Error",
    xaxis_title="CPU Time (hours)",
    yaxis_title=r"$\eta_{u,2}$",
    width=600,
    height=400,
    showlegend=False,
)
fig_cpu_pos.show()
if write:
    fig_cpu_pos.write_image(
        "../../output/figures/impact_position_error_vs_cpu_time_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

In [ ]:
# --- Plot 5: Velocity Error vs CPU Time ---
write = False

fig_cpu_vel = go.Figure()
add_traces_to_fig(
    fig_cpu_vel, calculated_data, y_key="vel_err", x_key="cpu_time", show_dt_lines=False
)
fig_cpu_vel.update_layout(get_layout())
fig_cpu_vel.update_xaxes(
    axis_style, type="log"
)  # Usually linear for time, but user can change to "log" if desired
fig_cpu_vel.update_yaxes(axis_style, type="log", side="right")
fig_cpu_vel.update_layout(
    title="CPU Time vs Velocity Error",
    xaxis_title="CPU Time (hours)",
    yaxis_title=r"$\eta_{v,2}$",
    width=600,
    height=400,
    showlegend=False,
)
fig_cpu_vel.show()
if write:
    fig_cpu_vel.write_image(
        "../../output/figures/impact_velocity_error_vs_cpu_time_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

In [ ]:
# --- Plot 6: Instability vs CPU Time ---
write = False

fig_cpu_instability = go.Figure()
add_traces_to_fig(
    fig_cpu_instability,
    calculated_data,
    y_key="instability",
    x_key="cpu_time",
    show_dt_lines=False,
)
fig_cpu_instability.update_layout(get_layout())
fig_cpu_instability.update_xaxes(
    axis_style, type="log"
)  # Usually linear for time, but user can change to "log" if desired
fig_cpu_instability.update_yaxes(axis_style, type="log", side="right")
fig_cpu_instability.update_layout(
    title="CPU Time vs Energy Instability",
    xaxis_title="CPU Time (hours)",
    yaxis_title=r"$\max(|\Delta E / E_{\text{int,max}}|)$",
    width=600,
    height=400,
    showlegend=False,
)
fig_cpu_instability.show()
if write:
    fig_cpu_instability.write_image(
        "../../output/figures/impact_energy_instability_vs_cpu_time_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

Box size runs

In [ ]:
run_names = [
    "output/cluster/box_size_study_r1e0_l2e-2_md2e6/BD-ns_frag1d_box-runs/run-1/nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf1.00e+00_seed1",
    "output/cluster/box_size_study_r1e0_l2e-2_md2e6/BD-ns_frag1d_box-runs/run-2/nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf2.00e+00_seed1",
    "output/cluster/box_size_study_r1e0_l2e-2_md2e6/BD-ns_frag1d_box-runs/run-3/nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf4.00e+00_seed1",
    "output/cluster/box_size_study_r1e0_l2e-2_md2e6/BD-ns_frag1d_box-runs/run-4/nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf6.00e+00_seed1",
    "output/cluster/box_size_study_r1e0_l2e-2_md2e6/BD-ns_frag1d_box-runs/run-5/nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf8.00e+00_seed1",
    "output/cluster/box_size_study_r1e0_l2e-2_md2e6/BD-ns_frag1d_box-runs/run-6/nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf1.00e+01_seed1",
    "output/cluster/box_size_study_r1e0_l2e-2_md2e6/BD-ns_frag1d_box-runs/run-11/nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf2.00e+01_seed1",
    "output/cluster/box_size_study_r1e0_l2e-2_md2e6/BD-ns_frag1d_box-runs/run-14/nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf4.00e+01_seed1",
    "output/cluster/box_size_study_r1e0_l2e-2_md2e6/BD-ns_frag1d_box-runs/run-16/nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf6.00e+01_seed1",
    "output/cluster/box_size_study_r1e0_l2e-2_md2e6/BD-ns_frag1d_box-runs/run-18/nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf8.00e+01_seed1",
    "output/cluster/box_size_study_r1e0_l2e-2_md2e6/BD-ns_frag1d_box-runs/run-20/nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1",
]
"mesh_convergence_r1e0_l5e-3_bf100"

colorscale = "Magenta"
line_colors = sample_colorscale(
    colorscale, len(run_names) + 1
)  # +1 to adjust last color
# line_colors = line_colors[2:]

param_regex = re.compile(r"([a-z]+)([-+]?[\d.]+(?:e[-+]?\d+)?)")
order_by = "bf"
bf_values = [
    dict((k, float(v)) for k, v in param_regex.findall(Path(rn).name)).get(
        order_by, np.nan
    )
    for rn in run_names
]

for run_name in run_names:
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    sim_time = df_run["simulation_time"].iloc[0]
    print(
        f"Simulation Time (hours): {sim_time/3600:.2f} for run: {run_name.split('/')[-1]}"
    )

axis_style = dict(
    showgrid=True,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    exponentformat="power",
    showexponent="last",
)

Simulation Time (hours): 62.14 for run: nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf1.00e+00_seed1
Simulation Time (hours): 76.45 for run: nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf2.00e+00_seed1
Simulation Time (hours): 0.00 for run: nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf4.00e+00_seed1
Simulation Time (hours): 91.79 for run: nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf6.00e+00_seed1
Simulation Time (hours): 92.39 for run: nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_bc_box_bf8.00e+00_seed1
Simulation Time (hours): 0.00 for run: nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.00e+00_sc-1.00e+01_

In [31]:
write = True

fig_edis = go.Figure()
for run_name, line_color in zip(run_names, line_colors):
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    fig_edis.add_trace(
        go.Scatter(
            x=df_run["time"],
            y=df_run["dissipated_energy"]
            / df_run["algorithmic_energy_balance"].iloc[0],
            mode="lines",
            line=dict(color=line_color, width=2),
            name=run_name.split("/")[-1],
        ),
    )
fig_edis.update_layout(get_layout())
fig_edis.update_xaxes(axis_style)
fig_edis.update_yaxes(axis_style)
fig_edis.update_layout(
    xaxis_title=r"$t$ (s)",
    yaxis_title=r"$\Delta \mathcal{G}/\mathcal{H}_0$",
    width=600,
    height=400,
    showlegend=False,
)
fig_edis.show()

if write:
    fig_edis.write_image(
        "../../output/figures/box_size_study_dissipated_energy_plot.pdf",
        scale=1,
        width=400,
        height=250,
    )

In [ ]:
write = True

fig_edis_final = go.Figure()
for run_name, line_color in zip(run_names, line_colors):
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    fig_edis_final.add_trace(
        go.Scatter(
            x=[bf_values[run_names.index(run_name)]],
            y=[
                df_run["dissipated_energy"].iloc[-1]
                / df_run["algorithmic_energy_balance"].iloc[0]
            ],
            mode="markers",
            marker=dict(size=8, color=line_color),
        ),
    )
fig_edis_final.update_layout(get_layout())
fig_edis_final.update_xaxes(axis_style, type="log")
fig_edis_final.update_yaxes(axis_style)
fig_edis_final.update_layout(
    xaxis_title=r"$\alpha_\mathrm{box}$",
    yaxis_title=r"$\Delta \mathcal{G}_\mathrm{final}/\mathcal{H}_0$",
    width=600,
    height=400,
    showlegend=False,
)
fig_edis_final.show()

if write:
    fig_edis_final.write_image(
        "../../output/figures/box_size_study_final_dissipated_energy_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

In [33]:
write = False

fig_nfrag = go.Figure()
for run_name, line_color in zip(run_names, line_colors):
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    fig_nfrag.add_trace(
        go.Scatter(
            x=df_run["time"],
            y=df_run["nb_fragments"],
            mode="lines",
            line=dict(color=line_color, width=2),
            name=run_name.split("/")[-1],
        ),
    )
fig_nfrag.update_layout(get_layout())
fig_nfrag.update_xaxes(axis_style)
fig_nfrag.update_yaxes(axis_style)
fig_nfrag.update_layout(
    xaxis_title=r"$t$ (s)",
    yaxis_title=r"$N_f$",
    width=600,
    height=400,
    showlegend=False,
)
fig_nfrag.show()
if write:
    fig_nfrag.write_image(
        "../../output/figures/box_size_study_number_of_fragments_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

In [34]:
write = False

fig_nfrag_final = go.Figure()
for run_name, line_color in zip(run_names, line_colors):
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    fig_nfrag_final.add_trace(
        go.Scatter(
            x=[bf_values[run_names.index(run_name)]],
            y=[df_run["nb_fragments"].iloc[-1]],
            mode="markers",
            marker=dict(size=10, color=line_color),
        ),
    )
fig_nfrag_final.update_layout(get_layout())
fig_nfrag_final.update_xaxes(axis_style)
fig_nfrag_final.update_yaxes(axis_style)
fig_nfrag_final.update_layout(
    xaxis_title=r"$\alpha_\mathrm{box}$",
    yaxis_title=r"$N_f$ at Final Time",
    width=600,
    height=400,
    showlegend=False,
)
fig_nfrag_final.show()

if write:
    fig_nfrag_final.write_image(
        "../../output/figures/box_size_study_number_of_fragments_final_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

### Length study (constrained expansion)

In [ ]:
run_names = [
    # "output/cluster/length_convergence_r1e0_md1e6_bf100/BD-ns_frag1d_box-runs/run-259/nsnfrag1d_l1.00e-03_n1.00e+03_md1.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1",
    # "output/cluster/length_convergence_r1e0_md1e6_bf100/BD-ns_frag1d_box-runs/run-260/nsnfrag1d_l2.00e-03_n2.00e+03_md1.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1",
    # "output/cluster/length_convergence_r1e0_md1e6_bf100/BD-ns_frag1d_box-runs/run-261/nsnfrag1d_l3.00e-03_n3.00e+03_md1.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1",
    # "output/cluster/length_convergence_r1e0_md1e6_bf100/BD-ns_frag1d_box-runs/run-262/nsnfrag1d_l4.00e-03_n4.00e+03_md1.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1",
    # "output/cluster/length_convergence_r1e0_md1e6_bf100/BD-ns_frag1d_box-runs/run-258/nsnfrag1d_l5.00e-03_n5.00e+03_md1.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1"
    "output/cluster/length_study_r1e0_bf100_md2e6/BD-ns_frag1d_box-runs/run-182/nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1",
    "output/cluster/length_study_r1e0_bf100_md2e6/BD-ns_frag1d_box-runs/run-183/nsnfrag1d_l2.00e-03_n4.00e+03_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1",
    "output/cluster/length_study_r1e0_bf100_md2e6/BD-ns_frag1d_box-runs/run-184/nsnfrag1d_l4.00e-03_n8.00e+03_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1",
    "output/cluster/length_study_r1e0_bf100_md2e6/BD-ns_frag1d_box-runs/run-185/nsnfrag1d_l6.00e-03_n1.20e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1",
    "output/cluster/length_study_r1e0_bf100_md2e6/BD-ns_frag1d_box-runs/run-186/nsnfrag1d_l8.00e-03_n1.60e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1",
    "output/cluster/length_study_r1e0_bf100_md2e6/BD-ns_frag1d_box-runs/run-187/nsnfrag1d_l1.00e-02_n2.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1",
    # "output/cluster/length_study_r1e0_bf100_md2e6/BD-ns_frag1d_box-runs/run-181/nsnfrag1d_l2.00e-02_n4.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1"
]

colorscale = "ice"
line_colors = sample_colorscale(
    colorscale, len(run_names) + 1
)  # +1 to adjust last color
# line_colors = line_colors[2:]

param_regex = re.compile(r"([a-z]+)([-+]?[\d.]+(?:e[-+]?\d+)?)")
order_by = "l"
normalize_by = "l"
l_values = [
    dict((k, float(v)) for k, v in param_regex.findall(Path(rn).name)).get(
        order_by, np.nan
    )
    for rn in run_names
]

for run_name in run_names:
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    sim_time = df_run["simulation_time"].iloc[0]
    print(
        f"Simulation Time (hours): {sim_time/3600:.2f} for run: {run_name.split('/')[-1]}"
    )

axis_style = dict(
    showgrid=True,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    exponentformat="power",
    showexponent="last",
)

mat = dict(E=370e9, Gc=50, sigma_c=262e6, rho=3900)
s0 = mat["E"] * mat["Gc"] / mat["sigma_c"] ** 2
U0 = mat["sigma_c"] ** 2 / mat["E"]

Simulation Time (hours): 1.65 for run: nsnfrag1d_l1.00e-03_n2.00e+03_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1
Simulation Time (hours): 8.80 for run: nsnfrag1d_l2.00e-03_n4.00e+03_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1
Simulation Time (hours): 25.33 for run: nsnfrag1d_l4.00e-03_n8.00e+03_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1
Simulation Time (hours): 38.94 for run: nsnfrag1d_l6.00e-03_n1.20e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1
Simulation Time (hours): 49.43 for run: nsnfrag1d_l8.00e-03_n1.60e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.00e+01_bc_box_bf1.00e+02_seed1
Simulation Time (hours): 74.32 for run: nsnfrag1d_l1.00e-02_n2.00e+04_md2.00e+06_p1_t2.00e-04_r1.00e+00_s9.90e-01_dd1.00e+05_e1.000e+00_sc-1.0

In [6]:
write = True

### X, Y and fit for total dissipated energy
# 1. Prepare the data for fitting
x_data_1 = np.array(l_values)
y_data_1 = np.array(
    [
        load_simulation_h5("../../" + r + "/data.h5")["dissipated_energy"].iloc[-1]
        / l
        / U0
        for r, l in zip(run_names, l_values)
    ]
)

# 2. Perform linear regression in log-log space: log10(y) = k * log10(x) + log10(a)
log_x = np.log10(x_data_1)
log_y = np.log10(y_data_1)
coeffs = np.polyfit(log_x, log_y, 1)
exponent = coeffs[0]
intercept = coeffs[1]

# 3. Generate the fit line for plotting
x_fit_1 = np.geomspace(x_data_1.min(), x_data_1.max(), 100)
y_fit_1 = 10**intercept * x_fit_1**exponent

### X, Y and fit for dissipated in complete cracks (1/s)
G_c = 50  # J/m^2
x_data_2 = np.array(l_values)
y_data_2 = np.array(
    [
        load_simulation_h5("../../" + r + "/data.h5")["nb_fragments"].iloc[-1]
        * mat["Gc"]
        / l
        / U0
        for r, l in zip(run_names, l_values)
    ]
)
log_x2 = np.log10(x_data_2)
log_y2 = np.log10(y_data_2)
coeffs2 = np.polyfit(log_x2, log_y2, 1)
exponent2 = coeffs2[0]
intercept2 = coeffs2[1]
x_fit_2 = np.geomspace(x_data_2.min(), x_data_2.max(), 100)
y_fit_2 = 10**intercept2 * x_fit_2**exponent2

# --- Plotting ---
fig = go.Figure()

# Add original markers
for i, run_name in enumerate(run_names):
    fig.add_trace(
        go.Scatter(
            x=[l_values[i]],
            y=[y_data_1[i]],
            mode="markers",
            marker=dict(size=8, color=line_colors[i]),
            name=f"L={l_values[i]:.2e} m",
            showlegend=True,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[l_values[i]],
            y=[y_data_2[i]],
            mode="markers",
            marker=dict(size=8, color=line_colors[i]),
            name=f"L={l_values[i]:.2e} m (Fragments)",
            showlegend=True,
        )
    )

# Add the fit line
fig.add_trace(
    go.Scatter(
        x=x_fit_1,
        y=y_fit_1,
        mode="lines",
        line=dict(dash="solid", color="black", width=1),
        name=f"Fit (Slope/Exponent: {exponent:.2f})",
    )
)

fig.add_trace(
    go.Scatter(
        x=x_fit_2,
        y=y_fit_2,
        mode="lines",
        line=dict(dash="solid", color="gray", width=1),
        name=f"Fit Fragments (Slope/Exponent: {exponent2:.2f})",
    )
)

fig.update_layout(get_layout())
fig.update_layout(
    xaxis=dict(type="log", title=r"$L \text(m)$", tickfont=dict(size=10), **axis_style),
    yaxis=dict(
        type="log", title=r"$\hat{\mathcal{G}}/L$", tickfont=dict(size=10), **axis_style
    ),
    title=f"Final Dissipated Energy vs Length (Exponent: {exponent:.2f})",
)
fig.show()
if write:
    fig.write_image(
        "../../output/figures/length_convergence_final_dissipated_energy_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

In [ ]:
# Plot dissipated energy / (number of fragments * Gc) as a function of
write = True

fig2 = go.Figure()

x_data_1 = np.array(l_values)
y_data_1 = np.array(
    [
        load_simulation_h5("../../" + r + "/data.h5")["dissipated_energy"].iloc[-1]
        for r, l in zip(run_names, l_values)
    ]
)

G_c = 50  # J/m^2
x_data_2 = np.array(l_values)
y_data_2 = np.array(
    [
        load_simulation_h5("../../" + r + "/data.h5")["nb_fragments"].iloc[-1]
        * mat["Gc"]
        for r, l in zip(run_names, l_values)
    ]
)

for i, run_name in enumerate(run_names):
    fig2.add_trace(
        go.Scatter(
            x=[l_values[i]],
            y=[y_data_1[i] / y_data_2[i]],
            mode="markers",
            marker=dict(size=8, color=line_colors[i]),
            name=f"L={l_values[i]:.2e} m",
            showlegend=True,
        )
    )
fig2.update_layout(
    get_layout(),
    xaxis=dict(type="log", title=r"$L \text(m)$", tickfont=dict(size=10), **axis_style),
    yaxis=dict(
        type="linear",
        title=r"$\hat{\mathcal{G}}/(N \times G_c)$",
        tickfont=dict(size=12),
        **axis_style,
        side="left",
    ),
    template="plotly_white",
)
fig2.show()

if write:
    fig2.write_image(
        "../../output/figures/length_convergence_dissipated_energy_per_fragment_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

In [ ]:
# --- Setup for Fragment Size Distribution ---
fig_fsd = go.Figure()
write_fsd = False


# We will plot the distribution for each bar length
for i, (run_name, l) in enumerate(zip(run_names, l_values)):

    # Load the simulation data
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")

    # Extract the array of fragment masses at the final time step.
    final_masses = np.array(df_run["fragment_mass"].iloc[-1])
    final_sizes = final_masses / mat["rho"]
    mean_final_size = np.mean(final_sizes)
    nb_fragments = len(final_sizes)

    # Sort sizes in descending order to compute N(>s)
    final_sizes_sorted = np.sort(final_sizes)[::-1]

    # Create the cumulative count array: 1, 2, 3... up to total number of fragments
    N_greater_than = np.arange(1, len(final_sizes_sorted) + 1)

    # Add to plot
    fig_fsd.add_trace(
        go.Scatter(
            x=final_sizes_sorted / s0,
            y=N_greater_than,
            mode="markers+lines",
            line=dict(color=line_colors[i], width=2),
            marker=dict(size=5, color=line_colors[i]),
            name=f"L={l:.2e} m",
            showlegend=True,
        )
    )

    x_data = final_sizes_sorted / s0
    y_data = N_greater_than
    mask = (x_data < np.inf) & (x_data > 0)
    x_filtered = x_data[mask]
    y_filtered = y_data[mask]
    mean_size_normalized = mean_final_size / s0
    N0 = l / mean_final_size  # N_greater_than[-1]
    x_fit_line = np.linspace(x_filtered.min(), x_filtered.max(), 100)
    y_fit_line = N0 * np.exp(-x_fit_line / mean_size_normalized)
    fig_fsd.add_trace(
        go.Scatter(
            x=x_fit_line,
            y=y_fit_line,
            mode="lines",
            line=dict(dash="dot", color=line_colors[i], width=2),
            name=f"Exp Fit (Mean Size): $\mu={mean_size_normalized:.3f}$",
        )
    )


# Get the x and y data from the last iteration of the loop
# x_data = final_sizes_sorted / s0
# y_data = N_greater_than
#
## 1. Filter the data
# mask = (x_data < 2) & (x_data > 0)
# x_filtered = x_data[mask]
# y_filtered = y_data[mask]
#
# 2. Ensure we have enough points to actually fit a line
# if len(x_filtered) > 1:
#    # 3. Perform an exponential fit (MEP): y = A * exp(-x / mu)
#    # In semi-log space, this is linear: ln(y) = (-1/mu) * x + ln(A)
#    ln_y = np.log(y_filtered)
#
#    # np.polyfit returns the slope (index 0) and the ln(A) intercept (index 1)
#    # Note: We use x_filtered here, NOT np.log(x_filtered)
#    slope, ln_intercept = np.polyfit(x_filtered, ln_y, 1)
#
#    # Extract the physical parameters
#    A = np.exp(ln_intercept)
#    mu = -1.0 / slope  # This represents the mean normalized size <s_hat>
#
#    # 4. Generate a smooth line for plotting the fit
#    # For exponential fits, linspace is better than geomspace
#    x_fit_line = np.linspace(x_filtered.min(), x_filtered.max(), 50)
#    y_fit_line = A * np.exp(-x_fit_line / mu)
#
#    # 5. Add the fit line to the Plotly figure
#    fig_fsd.add_trace(
#        go.Scatter(
#            x=x_fit_line,
#            y=y_fit_line,
#            mode="lines",
#            line=dict(dash="solid", color="black", width=2),
#            name=f"MEP Fit: $\mu={mu:.3f}$",
#        )
#    )
# else:
#    print("Warning: Not enough points to perform an exponential fit.")

# Plot an exponential fit using the true mean size


fig_fsd.update_layout(
    get_layout(),
    xaxis=dict(
        type="linear", title=r"$\hat{s}$"
    ),  # range=[0, 1.2], tickfont=dict(size=10)),
    yaxis=dict(
        type="log", title=r"$N(>\hat{s})$", tickfont=dict(size=10), side="right"
    ),
    title="Fragment Size Distribution (MEP / Exponential)",
)

fig_fsd.show()

if write_fsd:
    fig_fsd.write_image(
        "../../output/figures/fragment_size_distribution_plot.pdf",
        scale=1,
        width=400,
        height=350,
    )

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import curve_fit

# --- Setup for Fragment Size Distribution ---
fig_fsd = go.Figure()
write_fsd = True

# Physics-based Log-CCDF Function
# We hardcode the signs so the solver looks for positive A, B
# def log_physics_ccdf(s, l1, l3, ln_Nf):
#    return ln_Nf - l1 * s - l3 * (s**3)


def log_physics_ccdf_fixed(s, l1, l3):
    return np.log(n_f) - l1 * s - l3 * (s**3)


for i, (run_name, l) in enumerate(zip(run_names, l_values)):
    # 1. Load the simulation data
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")

    # 2. Extract and Normalize Sizes
    final_masses = np.array(df_run["fragment_mass"].iloc[-1])
    final_sizes = final_masses / mat["rho"]
    n_f = len(final_sizes)
    mean_final_size = np.mean(final_sizes)

    # Sort for CCDF: N(>s)
    final_sizes_sorted = np.sort(final_sizes)[::-1]
    x_data = final_sizes_sorted / s0
    y_data = np.arange(1, n_f + 1)

    # 3. Filter for valid data
    mask = (x_data > 0) & (y_data > 0)
    x_filtered = x_data[mask]
    y_filtered = y_data[mask]
    ln_y_filtered = np.log(y_filtered)

    # --- Add Empirical Data Trace ---
    fig_fsd.add_trace(
        go.Scatter(
            x=x_filtered,
            y=y_filtered,
            mode="markers",
            marker=dict(size=5, color=line_colors[i]),
            name=f"Data L={l:.2e} m",
        )
    )

    # --- Standard Exponential Fit (Theoretical Mean) ---
    # This is your 'Ideal' line based purely on mass conservation
    mean_size_normalized = mean_final_size / s0
    x_fit_line = np.geomspace(x_filtered.min(), x_filtered.max(), 100)
    # Using Nf = l/mean_final_size as you preferred for the theoretical intercept
    y_ideal = (l / mean_final_size) * np.exp(-x_fit_line / mean_size_normalized)

    # fig_fsd.add_trace(go.Scatter(
    #    x=x_fit_line, y=y_ideal,
    #    mode="lines",
    #    line=dict(dash="solid", color=line_colors[i], width=1),
    #    name=f"Ideal Exp (L/mu)"
    # ))

    # --- Physics-Based Log-Space Fit ---
    # We allow ln_Nf to be a parameter, but seed it with the actual count
    p0_guesses = [1.0 / mean_size_normalized, 1e-6]
    bounds = ([0, 0], [np.inf, np.inf])

    try:
        popt, _ = curve_fit(
            log_physics_ccdf_fixed,
            x_filtered,
            ln_y_filtered,
            p0=p0_guesses,
            bounds=bounds,
        )
        l1_fit, l3_fit = popt

        # Generate Curve
        y_fit_physics = np.exp(log_physics_ccdf_fixed(x_fit_line, l1_fit, l3_fit))

        fig_fsd.add_trace(
            go.Scatter(
                x=x_fit_line,
                y=y_fit_physics,
                mode="lines",
                line=dict(dash="solid", color=line_colors[i], width=2),
                name=f"Phys-Fit (l1={l1_fit:.2f}, l3={l3_fit:.2e})",
            )
        )

        print(
            f"L={l:.2e} | l3={l3_fit:.2e} | l1={l1_fit:.3f} | ln(Nf)={np.log(n_f):.2f} | Nf={n_f:.1f} | Actual Nf={n_f}"
        )

    except Exception as e:
        print(f"Fit failed for L={l}: {e}")

# --- Layout ---
fig_fsd.update_layout(
    get_layout(),
    xaxis=dict(type="log", title=r"$\hat{s}$"),
    yaxis=dict(type="log", title=r"$N(>\hat{s})$", side="right"),
    title="Fragment Size Distribution: Corrected Log-Space Physics Fit",
    template="plotly_white",
    legend=dict(y=2),
    showlegend=False,
)

fig_fsd.show()
if write_fsd:
    fig_fsd.write_image(
        "../../output/figures/fragment_size_distribution_physics_fit_plot.pdf",
        scale=1,
        width=400,
        height=350,
    )

L=2.00e-03 | l3=1.22e+01 | l1=0.000 | ln(Nf)=2.94 | Nf=19.0 | Actual Nf=19
L=4.00e-03 | l3=5.79e+00 | l1=1.733 | ln(Nf)=3.87 | Nf=48.0 | Actual Nf=48
L=6.00e-03 | l3=6.30e+00 | l1=2.697 | ln(Nf)=4.47 | Nf=87.0 | Actual Nf=87
L=8.00e-03 | l3=5.54e-01 | l1=3.952 | ln(Nf)=4.79 | Nf=120.0 | Actual Nf=120
L=1.00e-02 | l3=2.74e-01 | l1=4.694 | ln(Nf)=5.27 | Nf=195.0 | Actual Nf=195


In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import curve_fit
from scipy.integrate import quad

# --- 1. Define the Rigorous Exact Model ---


def exact_log_ccdf(s_array, l1, l3, ln_Nf):
    """
    Fits the data to the exact integral of the MaxEnt PDF:
    p(s) = A * exp(-l1*s - l3*s^3)

    The CCDF is Nf * integral_s^inf p(x) dx
    """

    # Define the unnormalized kernel
    def kernel(x):
        return np.exp(-l1 * x - l3 * (x**3))

    # Calculate the normalization constant A^-1 (Integral from 0 to infinity)
    # We use a large upper bound or let quad handle it.
    # For s^3 decay, 10 * mean_size is usually plenty.
    total_area, _ = quad(kernel, 0, np.inf)

    # Calculate the CCDF for each s: Integral from s to infinity
    # This loop is called frequently by curve_fit.
    ccdf_values = []
    for s in s_array:
        area_above_s, _ = quad(kernel, s, np.inf)
        # Avoid log(0)
        p_greater_than_s = max(area_above_s / total_area, 1e-15)
        ccdf_values.append(ln_Nf + np.log(p_greater_than_s))

    return np.array(ccdf_values)


# --- 2. Main Simulation Loop ---

fig_fsd = go.Figure()

for i, (run_name, l) in enumerate(zip(run_names, l_values)):
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")

    # Extract and Normalize Sizes
    final_masses = np.array(df_run["fragment_mass"].iloc[-1])
    final_sizes = final_masses / mat["rho"]
    n_f = len(final_sizes)
    mean_s_norm = np.mean(final_sizes) / s0

    # Sort for Empirical CCDF
    final_sizes_sorted = np.sort(final_sizes)[::-1]
    x_data = final_sizes_sorted / s0
    y_data = np.arange(1, n_f + 1)

    # Filter valid data
    mask = x_data > 0
    x_filtered = x_data[mask]
    ln_y_filtered = np.log(y_data[mask])

    # Plot Raw Data
    fig_fsd.add_trace(
        go.Scatter(
            x=x_filtered,
            y=np.exp(ln_y_filtered),
            mode="markers",
            marker=dict(size=4, color=line_colors[i]),
            name=f"Data L={l:.1e}",
        )
    )

    # --- Fitting ---
    # Initial Guesses:
    # l1 ~ 1/mean, l3 ~ 1/(3*mean^3), ln_Nf ~ log(total_fragments)
    p0 = [1.0 / mean_s_norm, 1.0 / (3 * mean_s_norm**3), np.log(n_f)]
    bounds = ([0, 0, -np.inf], [np.inf, np.inf, np.inf])

    print(f"Fitting L={l:.2e}... (Numerical integration takes a moment)")
    try:
        popt, _ = curve_fit(
            exact_log_ccdf, x_filtered, ln_y_filtered, p0=p0, bounds=bounds
        )
        l1_f, l3_f, ln_Nf_f = popt

        # Generate smooth fit line
        x_fit = np.geomspace(x_filtered.min(), x_filtered.max(), 50)
        y_fit = np.exp(exact_log_ccdf(x_fit, l1_f, l3_f, ln_Nf_f))

        fig_fsd.add_trace(
            go.Scatter(
                x=x_fit,
                y=y_fit,
                mode="lines",
                line=dict(dash="dash", color=line_colors[i]),
                name=f"Rigorous Fit (λ₁={l1_f:.2f}, λ₃={l3_f:.2e})",
            )
        )

        print(f"Success: λ1={l1_f:.3f}, λ3={l3_f:.3e}")

    except Exception as e:
        print(f"Fit failed: {e}")

# --- Layout ---
fig_fsd.update_layout(
    get_layout(),
    xaxis_type="log",
    yaxis_type="log",
    title="Rigorous Mixed-Constraint MaxEnt Fit",
    xaxis_title=r"$\hat{s}$",
    yaxis_title=r"$N(>\hat{s})$",
    template="plotly_white",
    legend=dict(y=1.5),
)
fig_fsd.show()

Fitting L=2.00e-03... (Numerical integration takes a moment)
Success: λ1=0.000, λ3=8.079e+00
Fitting L=4.00e-03... (Numerical integration takes a moment)
Success: λ1=0.000, λ3=4.600e+00
Fitting L=6.00e-03... (Numerical integration takes a moment)
Success: λ1=0.476, λ3=6.049e+00
Fitting L=8.00e-03... (Numerical integration takes a moment)
Success: λ1=4.540, λ3=1.132e-12
Fitting L=1.00e-02... (Numerical integration takes a moment)
Success: λ1=3.771, λ3=6.615e-01


In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import curve_fit
from scipy.integrate import quad

# --- 0. Material & Physical Constants ---
rho = mat["rho"]
E = mat["E"]
sigma_c = mat["sigma_c"]
G_c = mat["Gc"]

# Calculate derived constants
c = np.sqrt(E / rho)
t_0 = (E * G_c) / (sigma_c**2 * c)
eps_dot = sigma_c / (E * t_0)

# Local cubic energy coefficient (Constant across all runs)
kappa = (rho * eps_dot**2) / 24.0

# --- 1. Main Simulation Loop ---
fig_fsd = go.Figure()


p0_current = [1.0 / G_c, 0.5, np.log(n_f)]

for i, (run_name, l) in enumerate(zip(run_names, l_values)):  # noqa E741
    # Load your data
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")

    # Extract and Normalize Sizes
    final_masses = np.array(df_run["fragment_mass"].iloc[-1])
    final_sizes = final_masses / rho  # Using physical rho here
    n_f = len(final_sizes)

    # Sort for Empirical CCDF
    final_sizes_sorted = np.sort(final_sizes)[::-1]
    x_data = final_sizes_sorted / s0
    y_data = np.arange(1, n_f + 1)

    # Filter valid data
    mask = x_data > 0
    x_filtered = x_data[mask]
    ln_y_filtered = np.log(y_data[mask])

    # Plot Raw Data
    fig_fsd.add_trace(
        go.Scatter(
            x=x_filtered,
            y=np.exp(ln_y_filtered),
            mode="markers",
            marker=dict(size=4, color=line_colors[i]),
            name=f"Data L={l:.1e}",
        )
    )

    # --- 2. Define the Mixed-Mode Model for this specific L ---

    # Global linear energy coefficient (Depends on L)
    chi = (rho * (l**2) * eps_dot**2) / 24.0

    def exact_log_ccdf_mixed(
        x_array: np.ndarray, lam: float, eta: float, ln_Nf: float
    ) -> np.ndarray:  # noqa: N803
        """
        Fits the data to the exact integral of the mixed-mode MaxEnt PDF:
        p(s) = C * exp[-lam * (eta * kappa * s^3 + (1 - eta) * chi * s)]
        Since x_array is normalized (x = s/s0), we substitute s = x * s0
        """
        # Pre-calculate the effective coefficients for the normalized input `x`
        chi_eff = chi * s0
        kappa_eff = kappa * (s0**3)

        def kernel(x_val: float) -> float:
            return np.exp(
                -lam * (eta * kappa_eff * (x_val**3) + (1.0 - eta) * chi_eff * x_val)
            )

        total_area, _ = quad(kernel, 0, np.inf)

        ccdf_values = []
        for x_val in x_array:
            area_above_x, _ = quad(kernel, x_val, np.inf)
            # Avoid log(0)
            p_greater = max(area_above_x / total_area, 1e-15)
            ccdf_values.append(ln_Nf + np.log(p_greater))

        return np.array(ccdf_values)

    # --- 3. Fitting ---
    # Bounds:
    # lam must be positive.
    # eta must be between 0 and 1.
    # ln_Nf is unbounded.
    bounds = ([0.0, 0.0, -np.inf], [np.inf, 1.0, np.inf])

    print(f"Fitting L={l:.2e} with mixed-mode constraint...")
    try:
        popt, _ = curve_fit(
            exact_log_ccdf_mixed,
            x_filtered,
            ln_y_filtered,
            p0=p0_current,
            bounds=bounds,
            maxfev=5000,
        )
        lam_f, eta_f, ln_Nf_f = popt  # noqa: N816

        p0_current = popt  # Update for next run

        # Generate smooth fit line
        x_fit = np.geomspace(x_filtered.min(), x_filtered.max(), 50)
        y_fit = np.exp(exact_log_ccdf_mixed(x_fit, lam_f, eta_f, ln_Nf_f))

        fig_fsd.add_trace(
            go.Scatter(
                x=x_fit,
                y=y_fit,
                mode="lines",
                line=dict(dash="dash", color=line_colors[i]),
                name=f"Mixed Fit (η={eta_f:.2f}, λ={lam_f:.1e})",
            )
        )

        print(f"Success: λ={lam_f:.4e}, η={eta_f:.4f}")

    except Exception as e:
        print(f"Fit failed: {e}")

# --- 4. Layout ---
fig_fsd.update_layout(
    get_layout(),
    xaxis_type="log",
    yaxis_type="log",
    title="Coupled Mixed-Mode Energy MaxEnt Fit (η Partition)",
    xaxis_title="Normalized Size (s/s0)",
    yaxis_title="N(>s)",
    template="plotly_white",
    legend=dict(y=1.5),
)
fig_fsd.show()

Fitting L=2.00e-03 with mixed-mode constraint...
Success: λ=3.8780e+00, η=1.0000
Fitting L=4.00e-03 with mixed-mode constraint...
Success: λ=2.2078e+00, η=1.0000
Fitting L=6.00e-03 with mixed-mode constraint...
Success: λ=2.9038e+00, η=0.9998
Fitting L=8.00e-03 with mixed-mode constraint...
Success: λ=2.4731e-03, η=0.0000
Fitting L=1.00e-02 with mixed-mode constraint...
Success: λ=3.1883e-01, η=0.9959


In [10]:
import numpy as np
import plotly.graph_objects as go
import os

# --- Setup & Initialization ---
write_fsd = True
output_dir = "../../output/figures/video2/"

# Ensure the output directory exists
if write_fsd and not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 1. Pre-load data and find global axis bounds
dfs = []
global_min_size = np.inf
global_max_size = -np.inf
global_max_n = 1

print("Loading data and calculating global axis limits...")
for run_name in run_names:
    df_run = load_simulation_h5("../../" + run_name + "/data.h5")
    dfs.append(df_run)

    # Scan all steps to find the global min/max mass and max count
    for step_masses in df_run["fragment_mass"]:
        mass_array = np.array(step_masses)
        size_array = (
            mass_array / mat["rho"]
        )  # If you want to convert mass to size, otherwise skip this step
        if len(size_array) > 0:
            global_min_size = min(global_min_size, np.min(size_array))
            global_max_size = max(global_max_size, np.max(size_array))
            global_max_n = max(global_max_n, len(size_array))

# Calculate Plotly log scale ranges (requires log10 values)
# Padding is added (* 0.8 and * 1.5) so data doesn't touch the very edge of the plot
x_range_log = [
    np.log10(global_min_size * 0.8 / s0),
    np.log10(global_max_size * 1.5 / s0),
]
y_range_log = [np.log10(0.8), np.log10(global_max_n * 1.5)]

# --- Prepare Theoretical Reference Lines (Calculated once) ---
s_ref = np.geomspace(global_min_size, global_max_size, 50)
theory_y1 = (s_ref / s_ref[0]) ** (-1) * global_max_n * 0.1
theory_y05 = (s_ref / s_ref[0]) ** (-0.5) * global_max_n * 0.1
theory_y025 = (s_ref / s_ref[0]) ** (-0.25) * global_max_n * 0.1

# --- Loop Through Each Step ---
num_steps = max(len(df) for df in dfs)

print(f"Generating plots for {num_steps} steps...")
for step in range(num_steps):
    fig_fsd = go.Figure()

    # Plot data for each bar length at the current step
    for i, (df_run, l) in enumerate(zip(dfs, l_values)):

        # 2. Clamp the step index to avoid the IndexError
        # If 'step' is larger than the dataframe, it just grabs the last row
        safe_step = min(step, len(df_run) - 1)

        masses = np.array(df_run["fragment_mass"].iloc[safe_step])
        sizes = (
            masses / mat["rho"]
        )  # If you want to convert mass to size, otherwise skip this step

        # Skip tracing if the bar hasn't fragmented yet
        if len(sizes) == 0:
            continue

        sizes_sorted = np.sort(sizes)[::-1]
        N_greater_than = np.arange(1, len(sizes_sorted) + 1)

        fig_fsd.add_trace(
            go.Scatter(
                x=sizes_sorted / s0,
                y=N_greater_than,
                mode="markers",
                line=dict(color=line_colors[i], width=2),
                marker=dict(size=5, color=line_colors[i]),
                name=f"L={l:.2e} m",
                showlegend=True,
            )
        )

    # Add Theoretical Reference Lines
    # fig_fsd.add_trace(go.Scatter(x=s_ref, y=theory_y1, mode="lines",
    #                             line=dict(dash="solid", color="black", width=2), name="Theory: $N(>s) \propto s^{-1}$"))
    # fig_fsd.add_trace(go.Scatter(x=s_ref, y=theory_y05, mode="lines",
    #                             line=dict(dash="solid", color="black", width=2), name="Theory: $N(>s) \propto s^{-0.5}$"))
    # fig_fsd.add_trace(go.Scatter(x=s_ref, y=theory_y025, mode="lines",
    #                             line=dict(dash="solid", color="black", width=2), name="Theory: $N(>s) \propto s^{-0.25}$"))

    # --- Formatting & Fixed Axes ---
    fig_fsd.update_layout(
        get_layout(),  # Uncomment if using your custom layout function
        xaxis=dict(
            type="linear",
            title=r"Fragment Size proxy (Mass) $s$",
            tickfont=dict(size=10),
            range=[0, 1.5],  # <-- Lock X axis
        ),
        yaxis=dict(
            type="log",
            title=r"Cumulative Number of Fragments $N(>s)$",
            tickfont=dict(size=10),
            range=y_range_log,  # <-- Lock Y axis
        ),
        title=f"Fragment Size Distribution (CCDF) - Step {step}",
    )

    # Show only the first plot to avoid crashing the browser with hundreds of tabs
    # if step == 0:
    #    fig_fsd.show()

    # Save each step individually
    if write_fsd:
        # Zero-pad the filename (e.g. step_0001.pdf) so they sort correctly
        file_path = os.path.join(output_dir, f"fsd_plot_step_{step:04d}.png")
        fig_fsd.write_image(
            file_path,
            scale=1,
            width=500,
            height=500,
        )

Loading data and calculating global axis limits...
Generating plots for 671 steps...


KeyboardInterrupt: 

In [ ]:
write = False

# Plot the mean fragment size at final time vs length
x_data = np.array(l_values)
y_data_frag = np.array(
    [
        load_simulation_h5("../../" + r + "/data.h5")["nb_fragments"].iloc[-1]
        for r in run_names
    ]
)
mean_fragment_size = (l_values / y_data_frag) / s0

# 2. Perform linear regression in log-log space: log10(y) = k * log10(x) + log10(a)
log_x = np.log10(x_data)
log_y = np.log10(
    mean_fragment_size
)  # Exclude the last point if it's an outlier or not relevant for fitting
coeffs = np.polyfit(log_x, log_y, 1)
exponent = coeffs[0]
intercept = coeffs[1]

# 3. Generate the fit line for plotting
x_fit = np.geomspace(x_data.min(), x_data.max(), 100)
y_fit = 10**intercept * x_fit**exponent

fig_frag = go.Figure()
for i, run_name in enumerate(run_names):
    fig_frag.add_trace(
        go.Scatter(
            x=[l_values[i]],
            y=[mean_fragment_size[i]],
            mode="markers",
            marker=dict(size=8, color=line_colors[i]),
            name=f"L={l_values[i]:.2e} m",
            showlegend=True,
        )
    )

fig_frag.add_trace(
    go.Scatter(
        x=x_fit,
        y=y_fit,
        mode="lines",
        line=dict(dash="solid", color="black", width=1),
        name=f"Fit (Slope/Exponent: {exponent:.2f})",
    )
)
fig_frag.update_layout(get_layout())
fig_frag.update_layout(
    xaxis=dict(type="log", title=r"$L \text(m)$", tickfont=dict(size=10), **axis_style),
    yaxis=dict(type="log", title=r"$\hat{s}$", tickfont=dict(size=12), **axis_style),
    title="Mean Fragment Size at Final Time vs Length",
)
fig_frag.show()
if write:
    fig_frag.write_image(
        "../../output/figures/length_study_mean_fragment_size_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

### Restitution study

In [22]:
# study_name = "restitution_study_r1e0_bf100_md2e6"
study_name = "restitution_r1e0_md1e6_l5e-3_bf100"
study_path = "../../output/cluster/" + study_name + "/"

data = get_simulation_data(study_path=study_path, order_by="e")
df_run = load_simulation_h5(data["path"].iloc[0])

axis_style = dict(
    showgrid=True,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    exponentformat="power",
    showexponent="last",
)

colorscale = "Brwnyl_r"
n_colors = len(data)
line_colors = sample_colorscale(colorscale, n_colors)
line_colors[len(data) - 1] = "black"  # Override last color for NaN case

# find the min time across all runs to set a common x-axis limit
min_time = min(
    load_simulation_h5(row["path"])["time"].iloc[-1] for _, row in data.iterrows()
)
min_step = []
for _, row in data.iterrows():
    df = load_simulation_h5(row["path"])
    idx = np.searchsorted(df["time"], min_time, side="right") - 1
    min_step.append(max(0, idx))

print(f"Minimum simulation time across all runs: {min_time:.2e} seconds")
print(f"Minimum step index across all runs: {min_step}")

Minimum simulation time across all runs: 2.78e-05 seconds
Minimum step index across all runs: [np.int64(600), np.int64(600), np.int64(600), np.int64(600), np.int64(600), np.int64(600), np.int64(600), np.int64(600), np.int64(600), np.int64(600), np.int64(600), np.int64(600), np.int64(600)]


In [ ]:
# Plot the number of fragments over time for all runs
write = False

fig_nfrag_time = go.Figure()
for (_, row), color in zip(data.iterrows(), line_colors):
    df_run = load_simulation_h5(row["path"])
    fig_nfrag_time.add_trace(
        go.Scatter(
            x=df_run["time"],
            y=df_run["nb_fragments"],
            mode="lines",
            line=dict(width=2, color=color),
            name=f"e={row['e']:.2f}" if pd.notna(row["e"]) else "nonsmooth",
        ),
    )
fig_nfrag_time.update_layout(get_layout())
fig_nfrag_time.update_xaxes(axis_style)
fig_nfrag_time.update_yaxes(axis_style)
fig_nfrag_time.update_layout(
    xaxis_title="Time (s)",
    yaxis_title=r"$N_f$",
    width=600,
    height=400,
    showlegend=False,
    yaxis=dict(side="right"),
    xaxis=dict(range=[0, min_time]),
)
fig_nfrag_time.show()
if write:
    fig_nfrag_time.write_image(
        f"../../output/figures/{study_name}_number_of_fragments_over_time_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

In [ ]:
write = False
ticks = [0, 0.5, 0.9, 0.99, 0.999, 0.9999, 1]
tick_vals = [(1 - t if t < 1 else 1e-5) for t in ticks]
tick_text = [str(t) for t in ticks]

fig_nfrag_final = go.Figure()

for (_, row), color, step in zip(data.iterrows(), line_colors, min_step):
    df_run = load_simulation_h5(row["path"])
    val_e = row["e"]
    if pd.isna(val_e):
        continue

    # Map e to distance from 1 (with a floor for e=1)
    plot_x = 1 - val_e if val_e < 1 else 1e-5

    fig_nfrag_final.add_trace(
        go.Scatter(
            x=[plot_x],
            y=[df_run["nb_fragments"].iloc[step]],
            mode="markers",
            marker=dict(size=8, color=color),
            name=f"e={val_e:.4f}",
        ),
    )

# Restore your specific Y-axis and X-axis styling
fig_nfrag_final.update_layout(get_layout())
fig_nfrag_final.update_layout(
    xaxis=dict(
        type="log",
        autorange="reversed",
        tickvals=tick_vals,
        ticktext=tick_text,
        title=r"$e$",
        side="top",  # Original preference
    ),
    yaxis=dict(
        title=r"$\left(N_f\right)_\text{final}$",
        side="right",  # Original preference
    ),
    width=600,
    height=400,
    showlegend=False,
)

fig_nfrag_final.show()

if write:
    fig_nfrag_final.write_image(
        f"../../output/figures/{study_name}_number_of_fragments_final_plot_log.pdf",
        scale=1,
        width=400,
        height=250,
    )

In [ ]:
write = False

fig_nfrag_final = go.Figure()
for (_, row), color, step in zip(data.iterrows(), line_colors, min_step):
    df_run = load_simulation_h5(row["path"])
    fig_nfrag_final.add_trace(
        go.Scatter(
            x=[row["e"] if pd.notna(row["e"]) else np.nan],
            y=[df_run["nb_fragments"].iloc[step]],
            mode="markers",
            marker=dict(size=8, color=color),
            name=f"e={row['e']:.2f}" if pd.notna(row["e"]) else "nonsmooth",
        ),
    )
fig_nfrag_final.update_layout(get_layout())
fig_nfrag_final.update_xaxes(axis_style)
fig_nfrag_final.update_yaxes(axis_style)
fig_nfrag_final.update_layout(
    xaxis_title=r"$e$",
    yaxis_title=r"$\left(N_f\right)_\text{final}$",
    yaxis=dict(side="right"),
    xaxis=dict(side="top"),
    width=600,
    height=400,
    showlegend=False,
)

fig_nfrag_final.show()
if write:
    fig_nfrag_final.write_image(
        f"../../output/figures/{study_name}_number_of_fragments_final_plot.pdf",
        scale=1,
        width=400,
        height=250,
    )

In [ ]:
# Plot the number of fragments over time for all runs
write = False

fig_edis_time = go.Figure()
for (_, row), color in zip(data.iterrows(), line_colors):
    df_run = load_simulation_h5(row["path"])
    norm = (
        df_run["algorithmic_energy"].iloc[-1]
        + df_run["contact_dissipation"].iloc[-1]
        + df_run["dissipated_energy"].iloc[-1]
    )
    if row["e"] != 1:
        fig_edis_time.add_trace(
            go.Scatter(
                x=df_run["time"],
                y=df_run["dissipated_energy"] / norm,
                mode="lines",
                line=dict(width=2, color=color, dash="solid"),
                name=f"e={row['e']:.2f}" if pd.notna(row["e"]) else "nonsmooth",
            ),
        )
        fig_edis_time.add_trace(
            go.Scatter(
                x=df_run["time"],
                y=(df_run["dissipated_energy"] + df_run["contact_dissipation"]) / norm,
                mode="lines",
                line=dict(width=2, color=color, dash="solid"),
                name=f"e={row['e']:.2f}" if pd.notna(row["e"]) else "nonsmooth",
            ),
        )

    elif row["e"] == 1:
        fig_edis_time.add_trace(
            go.Scatter(
                x=df_run["time"],
                y=df_run["dissipated_energy"] / norm,
                mode="lines",
                line=dict(width=2, color=color),
                name=f"e={row['e']:.2f}" if pd.notna(row["e"]) else "nonsmooth",
            ),
        )
fig_edis_time.update_layout(get_layout())
fig_edis_time.update_xaxes(axis_style)
fig_edis_time.update_yaxes(axis_style)
fig_edis_time.update_layout(
    xaxis_title="Time (s)",
    yaxis_title=r"$\mathcal{E}_\text{dis} / \mathcal{E}_\text{inj}$",
    width=600,
    height=400,
    showlegend=False,
    xaxis=dict(range=[0, min_time]),
)
fig_edis_time.show()
if write:
    fig_edis_time.write_image(
        f"../../output/figures/{study_name}_dissipated_energy_over_time_plot.pdf",
        scale=1,
        width=400,
        height=300,
    )

fig_edis_time.update_layout(
    width=600,
    height=400,
    showlegend=False,
    xaxis=dict(range=[0, 8e-6]),
    yaxis=dict(range=[0.1, 0.14]),
)
fig_edis_time.show()
if write:
    fig_edis_time.write_image(
        f"../../output/figures/{study_name}_dissipated_energy_over_time_plot_zoom.pdf",
        scale=1,
        width=280,
        height=250,
    )

In [ ]:
write = False

fig_edis_final = go.Figure()
for (_, row), color, step in zip(data.iterrows(), line_colors, min_step):
    df_run = load_simulation_h5(row["path"])
    norm = (
        df_run["algorithmic_energy"].iloc[-1]
        + df_run["contact_dissipation"].iloc[-1]
        + df_run["dissipated_energy"].iloc[-1]
    )
    if row["e"] != 1:
        fig_edis_final.add_trace(
            go.Scatter(
                x=[row["e"] if pd.notna(row["e"]) else np.nan],
                y=[df_run["dissipated_energy"].iloc[step] / norm],
                mode="markers",
                marker=dict(size=8, color=color, symbol="circle"),
                name=(
                    f"e={row['e']:.2f} (Dissipated)"
                    if pd.notna(row["e"])
                    else "nonsmooth"
                ),
            ),
        )
        fig_edis_final.add_trace(
            go.Scatter(
                x=[row["e"] if pd.notna(row["e"]) else np.nan],
                y=[
                    (
                        df_run["dissipated_energy"].iloc[step]
                        + df_run["contact_dissipation"].iloc[step]
                    )
                    / norm
                ],
                mode="markers",
                marker=dict(size=8, color=color, symbol="circle"),
                name=(
                    f"e={row['e']:.2f} (Total Dissipation)"
                    if pd.notna(row["e"])
                    else "nonsmooth"
                ),
            ),
        )
    elif row["e"] == 1:
        fig_edis_final.add_trace(
            go.Scatter(
                x=[row["e"] if pd.notna(row["e"]) else np.nan],
                y=[df_run["dissipated_energy"].iloc[step] / norm],
                mode="markers",
                marker=dict(size=8, color=color, symbol="circle"),
                name=(
                    f"e={row['e']:.2f} (Dissipated)"
                    if pd.notna(row["e"])
                    else "nonsmooth"
                ),
            ),
        )
fig_edis_final.update_layout(get_layout())
fig_edis_final.update_xaxes(axis_style)
fig_edis_final.update_yaxes(axis_style)
fig_edis_final.update_layout(
    xaxis_title=r"$e$",
    yaxis_title=r"$\left(\mathcal{E}_\text{dis} / \mathcal{E}_\text{inj}\right)_\text{final}$",
    xaxis=dict(side="top"),
    width=600,
    height=400,
    showlegend=False,
)
fig_edis_final.show()
if write:
    fig_edis_final.write_image(
        f"../../output/figures/{study_name}_final_dissipated_energy_plot.pdf",
        scale=1,
        width=400,
        height=250,
    )

In [ ]:
write = False
ticks = [0, 0.5, 0.9, 0.99, 0.999, 0.9999, 1]
tick_vals = [(1 - t if t < 1 else 1e-5) for t in ticks]
tick_text = [str(t) for t in ticks]

fig_edis_final = go.Figure()

for (_, row), color, step in zip(data.iterrows(), line_colors, min_step):
    df_run = load_simulation_h5(row["path"])
    norm = (
        df_run["algorithmic_energy"].iloc[-1]
        + df_run["contact_dissipation"].iloc[-1]
        + df_run["dissipated_energy"].iloc[-1]
    )

    val_e = row["e"]
    if pd.isna(val_e):
        continue

    # Map e to distance from 1 (with a floor for e=1)
    plot_x = 1 - val_e if val_e < 1 else 1e-5

    if row["e"] != 1:
        fig_edis_final.add_trace(
            go.Scatter(
                x=[plot_x],
                y=[df_run["dissipated_energy"].iloc[step] / norm],
                mode="markers",
                marker=dict(size=8, color=color, symbol="circle"),
                name=f"e={val_e:.4f} (Dissipated)",
            ),
        )
        fig_edis_final.add_trace(
            go.Scatter(
                x=[plot_x],
                y=[
                    (
                        df_run["dissipated_energy"].iloc[step]
                        + df_run["contact_dissipation"].iloc[step]
                    )
                    / norm
                ],
                mode="markers",
                marker=dict(size=8, color=color, symbol="circle"),
                name=f"e={val_e:.4f} (Total Dissipation)",
            ),
        )
    elif row["e"] == 1:
        fig_edis_final.add_trace(
            go.Scatter(
                x=[plot_x],
                y=[df_run["dissipated_energy"].iloc[step] / norm],
                mode="markers",
                marker=dict(size=8, color=color, symbol="circle"),
                name=f"e={val_e:.4f} (Dissipated)",
            ),
        )

# Restore your specific Y-axis and X-axis styling
fig_edis_final.update_layout(get_layout())
fig_edis_final.update_layout(
    xaxis=dict(
        type="log",
        autorange="reversed",
        tickvals=tick_vals,
        ticktext=tick_text,
        title=r"$e$",
        side="top",
    ),
    yaxis=dict(
        title=r"$\left(\mathcal{E}_\text{dis} / \mathcal{E}_\text{inj}\right)_\text{final}$",
        side="left",
    ),
    width=600,
    height=400,
    showlegend=False,
)

fig_edis_final.show()

if write:
    fig_edis_final.write_image(
        f"../../output/figures/{study_name}_final_dissipated_energy_plot_log.pdf",
        scale=1,
        width=400,
        height=250,
    )

In [24]:
# Plot as a function of e the ratio of dissipated energy to number of fragments at final time
write = True
G_c = 50  # J/m^2
ticks = [0, 0.5, 0.9, 0.99, 0.999, 0.9999, 1]
tick_vals = [(1 - t if t < 1 else 1e-5) for t in ticks]
tick_text = [str(t) for t in ticks]

fig_edis_per_frag_final = go.Figure()

for (_, row), color, step in zip(data.iterrows(), line_colors, min_step):
    df_run = load_simulation_h5(row["path"])
    val_e = row["e"]
    if pd.isna(val_e):
        continue

    # Map e to distance from 1 (with a floor for e=1)
    plot_x = 1 - val_e if val_e < 1 else 1e-5

    if row["e"] != 1:
        fig_edis_per_frag_final.add_trace(
            go.Scatter(
                x=[plot_x],
                y=[
                    df_run["dissipated_energy"].iloc[step]
                    / (df_run["nb_fragments"].iloc[step] * G_c)
                ],
                mode="markers",
                marker=dict(size=8, color=color, symbol="circle"),
                name=f"e={val_e:.4f}",
            ),
        )
    elif row["e"] == 1:
        fig_edis_per_frag_final.add_trace(
            go.Scatter(
                x=[plot_x],
                y=[
                    df_run["dissipated_energy"].iloc[step]
                    / (df_run["nb_fragments"].iloc[step] * G_c)
                ],
                mode="markers",
                marker=dict(size=8, color=color, symbol="circle"),
                name=f"e={val_e:.4f}",
            ),
        )

# Restore your specific Y-axis and X-axis styling
fig_edis_per_frag_final.update_layout(get_layout())
fig_edis_per_frag_final.update_layout(
    xaxis=dict(tickvals=tick_vals, ticktext=tick_text)
)
fig_edis_per_frag_final.update_layout(
    xaxis=dict(
        type="log",
        autorange="reversed",
        tickvals=tick_vals,
        ticktext=tick_text,
        title=r"$e$",
        side="bottom",
    ),
    yaxis=dict(
        type="linear",
        title=r"$\mathcal{G} / (N_f \times G_c)$",
        # side="left",
    ),
    width=600,
    height=400,
    showlegend=False,
)
fig_edis_per_frag_final.show()
if write:
    fig_edis_per_frag_final.write_image(
        f"../../output/figures/{study_name}_final_dissipated_energy_per_fragment_plot_log.pdf",
        scale=1,
        width=400,
        height=350,
    )